In [2]:
import os
from pathlib import Path

# Define the project directory in Colab's local storage
BASE_DIR = Path("/content/AMR_Structural_Mutation_Project")

# Define the project subdirectories
SUBDIRECTORIES = [
    "raw",
    "processed",
    "analysis",
    "figures",
    "manuscript"
]

# Create the project directory and subdirectories
BASE_DIR.mkdir(parents=True, exist_ok=True)

for folder in SUBDIRECTORIES:
    folder_path = BASE_DIR / folder
    folder_path.mkdir(parents=True, exist_ok=True)
    print(f"Directory verified/created: {folder_path}")

print(f"\nProject directory: {BASE_DIR}")
print("Step 1 Complete: Project directory structure established.")

Directory verified/created: /content/AMR_Structural_Mutation_Project/raw
Directory verified/created: /content/AMR_Structural_Mutation_Project/processed
Directory verified/created: /content/AMR_Structural_Mutation_Project/analysis
Directory verified/created: /content/AMR_Structural_Mutation_Project/figures
Directory verified/created: /content/AMR_Structural_Mutation_Project/manuscript

Project directory: /content/AMR_Structural_Mutation_Project
Step 1 Complete: Project directory structure established.


In [3]:
import os
import urllib.request
import tarfile
from pathlib import Path

# Define the raw data directory
RAW_DIR = BASE_DIR / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Define the downloaded archive path
TAR_PATH = RAW_DIR / "card-data.tar.bz2"

# CARD dataset download URL
CARD_URL = "https://card.mcmaster.ca/latest/data"

# 1. Download the latest CARD dataset
print("Downloading CARD dataset from card.mcmaster.ca...")

try:
    urllib.request.urlretrieve(CARD_URL, TAR_PATH)

    print(
        f"Downloaded: {TAR_PATH} "
        f"({TAR_PATH.stat().st_size:,} bytes)"
    )

except Exception as error:
    if TAR_PATH.exists():
        TAR_PATH.unlink()
    raise RuntimeError(
        f"Failed to download the CARD dataset: {error}"
    ) from error

# 2. Extract the archive safely into the raw directory
print("\nExtracting CARD dataset...")

try:
    with tarfile.open(TAR_PATH, mode="r:bz2") as tar:
        # Prevent archive members from escaping the destination directory
        tar.extractall(path=RAW_DIR, filter="data")

    print("Extraction completed successfully.")

except Exception as error:
    raise RuntimeError(
        f"Failed to extract the CARD dataset: {error}"
    ) from error

# 3. List extracted files
print("\nExtracted CARD dataset files:")

for path in sorted(RAW_DIR.iterdir()):
    if path.is_file():
        print(f" - {path.name} ({path.stat().st_size:,} bytes)")
    elif path.is_dir():
        print(f" - {path.name}/ [directory]")

print("\nCARD dataset download and extraction complete.")

Downloaded: /content/AMR_Structural_Mutation_Project/raw/card-data.tar.bz2 (4,366,637 bytes)

Extracting CARD dataset...
Extraction completed successfully.

Extracted CARD dataset files:
 - CARD-Download-README.txt (4,024 bytes)
 - PMID.tsv (220,995 bytes)
 - aro_categories.tsv (37,884 bytes)
 - aro_categories_index.tsv (735,859 bytes)
 - aro_index.tsv (1,127,760 bytes)
 - card-data.tar.bz2 (4,366,637 bytes)
 - card.json (38,086,878 bytes)
 - nucleotide_fasta_protein_homolog_model.fasta (6,285,453 bytes)
 - nucleotide_fasta_protein_knockout_model.fasta (16,904 bytes)
 - nucleotide_fasta_protein_overexpression_model.fasta (11,562 bytes)
 - nucleotide_fasta_protein_variant_model.fasta (429,485 bytes)
 - nucleotide_fasta_rRNA_gene_variant_model.fasta (186,540 bytes)
 - protein_fasta_protein_homolog_model.fasta (2,332,067 bytes)
 - protein_fasta_protein_knockout_model.fasta (6,261 bytes)
 - protein_fasta_protein_overexpression_model.fasta (4,516 bytes)
 - protein_fasta_protein_variant_mode

In [4]:
import json
from pathlib import Path

# Define the path to card.json
RAW_DIR = BASE_DIR / "raw"
JSON_PATH = RAW_DIR / "card.json"

# Check that the CARD JSON file exists
if not JSON_PATH.exists():
    raise FileNotFoundError(
        f"Cannot find {JSON_PATH}. "
        "Please run the CARD download and extraction cell first."
    )

# Load card.json
with JSON_PATH.open("r", encoding="utf-8") as f:
    card_data = json.load(f)

print(f"Total keys in card.json: {len(card_data)}")

# Classify entry types and identify variant models
model_types = set()
variant_entries = []

for key, entry in card_data.items():
    if isinstance(entry, dict):
        m_type = entry.get("model_type")

        if m_type:
            model_types.add(m_type)

        if m_type and "variant" in m_type.lower():
            variant_entries.append(entry)

print("Discovered model types in CARD:", model_types)
print(
    f"Number of variant model entries found: "
    f"{len(variant_entries)}"
)

# Print structural details of a sample variant model
if variant_entries:
    sample = variant_entries[0]

    print("\nSample Variant Model Inspection:")
    print("ARO Accession:", sample.get("ARO_accession"))
    print("ARO Name:", sample.get("ARO_name"))
    print("Top-level keys:", list(sample.keys()))

    if "model_param" in sample:
        model_param = sample["model_param"]

        if isinstance(model_param, dict):
            print("model_param keys:", list(model_param.keys()))
        else:
            print("model_param type:", type(model_param).__name__)
else:
    print("No variant model entries were found.")

Total keys in card.json: 6454
Discovered model types in CARD: {'protein variant model', 'protein knockout model', 'gene cluster meta-model', 'protein homolog model', 'protein overexpression model', 'rRNA gene variant model', 'efflux pump system meta-model'}
Number of variant model entries found: 325

Sample Variant Model Inspection:
ARO Accession: 3003582
ARO Name: basR
Top-level keys: ['model_id', 'model_name', 'model_type', 'model_type_id', 'model_description', 'model_param', 'model_sequences', 'ARO_accession', 'ARO_id', 'ARO_name', 'CARD_short_name', 'ARO_description', 'ARO_category']
model_param keys: ['snp', 'blastp_bit_score']


In [5]:
import json
import pandas as pd
from itertools import combinations

# Define input and output paths
RAW_DIR = BASE_DIR / "raw"
PROCESSED_DIR = BASE_DIR / "processed"
JSON_PATH = RAW_DIR / "card.json"
OUT_PATH = PROCESSED_DIR / "candidate_pairs.csv"

# Ensure the output directory exists
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Check that the CARD JSON file exists
if not JSON_PATH.exists():
    raise FileNotFoundError(
        f"Cannot find {JSON_PATH}. "
        "Please run the CARD download and extraction cells first."
    )

# Load CARD data
with JSON_PATH.open("r", encoding="utf-8") as f:
    card_data = json.load(f)

candidate_pairs = []
pair_counter = 1

for model_key, entry in card_data.items():
    if not isinstance(entry, dict):
        continue

    m_type = entry.get("model_type", "")

    if not isinstance(m_type, str) or "variant" not in m_type.lower():
        continue

    model_param = entry.get("model_param", {})
    if not isinstance(model_param, dict):
        continue

    snp_param = model_param.get("snp", {})

    # Extract SNP list from model_param
    snps = []

    if isinstance(snp_param, dict):
        param_val = snp_param.get("param_value", {})

        if isinstance(param_val, dict):
            for key, val in param_val.items():
                if isinstance(val, list):
                    snps.extend(val)
                elif isinstance(val, str):
                    snps.append(val)

        elif isinstance(param_val, list):
            snps = param_val

    # Clean SNP list and retain unique mutation entries per model
    snps = sorted(set(str(s).strip() for s in snps if s))

    # Extract pairs if two or more mutations exist in the model
    if len(snps) >= 2:
        gene_name = entry.get("ARO_name", "Unknown")
        card_model = entry.get("model_name", "Unknown")

        # Extract organism and drug information from categories
        organism = "Unspecified"
        drug = "Unspecified"

        categories = entry.get("ARO_category", {})

        if isinstance(categories, dict):
            for cat_id, cat_info in categories.items():
                if not isinstance(cat_info, dict):
                    continue

                cat_name = cat_info.get("category_aro_name", "")

                if not isinstance(cat_name, str):
                    continue

                if "resistance mechanism" in cat_name.lower():
                    continue
                elif (
                    "drug" in cat_name.lower()
                    or "antibiotic" in cat_name.lower()
                ):
                    drug = cat_name
                else:
                    organism = cat_name

        # Generate pair-level records
        for m1, m2 in combinations(snps, 2):
            candidate_pairs.append({
                "Pair_id": f"PAIR_{pair_counter:04d}",
                "Protein/gene": gene_name,
                "Mutation_1": m1,
                "Mutation_2": m2,
                "Organism": organism,
                "Drug": drug,
                "CARD_model": card_model
            })

            pair_counter += 1

# Convert records to a DataFrame
columns = [
    "Pair_id",
    "Protein/gene",
    "Mutation_1",
    "Mutation_2",
    "Organism",
    "Drug",
    "CARD_model"
]

df_pairs = pd.DataFrame(candidate_pairs, columns=columns)

# Save the output
df_pairs.to_csv(OUT_PATH, index=False)

print(f"Total candidate mutation pairs extracted: {len(df_pairs)}")
print(f"Output saved to: {OUT_PATH}")

if not df_pairs.empty:
    print("\nSample candidate pairs:")
    print(df_pairs.head(10).to_string(index=False))
else:
    print(
        "No candidate mutation pairs were extracted. "
        "Check the CARD model structure and SNP extraction logic."
    )

Total candidate mutation pairs extracted: 121325
Output saved to: /content/AMR_Structural_Mutation_Project/processed/candidate_pairs.csv

Sample candidate pairs:
  Pair_id                                                              Protein/gene Mutation_1 Mutation_2                                            Organism                         Drug                                                                CARD_model
PAIR_0001                                                                      basR      L157Q       L71R                 pmr phosphoethanolamine transferase            antibiotic efflux                                                                      basR
PAIR_0002       Streptococcus pneumoniae PBP1a conferring resistance to amoxicillin      N609D      S575T                              penicillin beta-lactam antibiotic target alteration       Streptococcus pneumoniae PBP1a conferring resistance to amoxicillin
PAIR_0003 Escherichia coli folP with mutation conferrin

In [6]:
import re
import pandas as pd

# Define input and output paths
PROCESSED_DIR = BASE_DIR / "processed"
IN_PATH = PROCESSED_DIR / "candidate_pairs.csv"

USABLE_PATH = PROCESSED_DIR / "usable_pairs.csv"
EXCLUDED_PATH = PROCESSED_DIR / "excluded_pairs_step4.csv"

# Check that the input file exists
if not IN_PATH.exists():
    raise FileNotFoundError(
        f"Cannot find {IN_PATH}. "
        "Please run the candidate mutation pair extraction cell first."
    )

# Load candidate mutation pairs
df = pd.read_csv(IN_PATH)
candidate_count = len(df)

# Standard single-letter amino acid mutation pattern (WT, POS, MUT)
aa_pattern = re.compile(
    r"^([ACDEFGHIKLMNPQRSTVWY])(\d+)([ACDEFGHIKLMNPQRSTVWY])$"
)

usable_pairs = []
excluded_records = []

for idx, row in df.iterrows():
    m1 = str(row["Mutation_1"]).strip()
    m2 = str(row["Mutation_2"]).strip()
    pair_id = row["Pair_id"]

    match1 = aa_pattern.fullmatch(m1)
    match2 = aa_pattern.fullmatch(m2)

    if not match1 or not match2:
        reason = "Non-standard or non-amino acid mutation notation"

        excluded_records.append({
            "Pair_id": pair_id,
            "Reason_excluded": reason,
            "Mutation_1": m1,
            "Mutation_2": m2
        })
        continue

    wt1, pos1, mut1 = match1.groups()
    wt2, pos2, mut2 = match2.groups()

    pos1, pos2 = int(pos1), int(pos2)

    if pos1 == pos2:
        reason = "Same residue position (alternative single-point alleles)"

        excluded_records.append({
            "Pair_id": pair_id,
            "Reason_excluded": reason,
            "Mutation_1": m1,
            "Mutation_2": m2
        })
        continue

    row_dict = row.to_dict()

    row_dict["WT_1"] = wt1
    row_dict["Pos_1"] = pos1
    row_dict["Mut_1"] = mut1

    row_dict["WT_2"] = wt2
    row_dict["Pos_2"] = pos2
    row_dict["Mut_2"] = mut2

    usable_pairs.append(row_dict)

# Create DataFrames with consistent columns
df_usable = pd.DataFrame(usable_pairs)

excluded_columns = [
    "Pair_id",
    "Reason_excluded",
    "Mutation_1",
    "Mutation_2"
]

df_excluded = pd.DataFrame(
    excluded_records,
    columns=excluded_columns
)

# Save results
df_usable.to_csv(USABLE_PATH, index=False)
df_excluded.to_csv(EXCLUDED_PATH, index=False)

# Report summary statistics
print(f"Total candidate pairs evaluated: {candidate_count}")
print(f"Total usable pairs retained: {len(df_usable)}")
print(f"Total pairs excluded: {len(df_excluded)}")

print(f"\nUsable pairs saved to: {USABLE_PATH}")
print(f"Excluded pairs saved to: {EXCLUDED_PATH}")

print("\nBreakdown of exclusion reasons:")

if not df_excluded.empty:
    print(
        df_excluded["Reason_excluded"]
        .value_counts()
        .to_string()
    )
else:
    print("No pairs were excluded.")

if not df_usable.empty:
    print("\nSample retained usable pairs:")

    sample_columns = [
        "Pair_id",
        "Protein/gene",
        "WT_1",
        "Pos_1",
        "Mut_1",
        "WT_2",
        "Pos_2",
        "Mut_2"
    ]

    print(
        df_usable[sample_columns]
        .head(5)
        .to_string(index=False)
    )
else:
    print("\nNo usable mutation pairs remain after filtering.")

Total candidate pairs evaluated: 121325
Total usable pairs retained: 114068
Total pairs excluded: 7257

Usable pairs saved to: /content/AMR_Structural_Mutation_Project/processed/usable_pairs.csv
Excluded pairs saved to: /content/AMR_Structural_Mutation_Project/processed/excluded_pairs_step4.csv

Breakdown of exclusion reasons:
Reason_excluded
Non-standard or non-amino acid mutation notation            5934
Same residue position (alternative single-point alleles)    1323

Sample retained usable pairs:
  Pair_id                                                           Protein/gene WT_1  Pos_1 Mut_1 WT_2  Pos_2 Mut_2
PAIR_0001                                                                   basR    L    157     Q    L     71     R
PAIR_0002    Streptococcus pneumoniae PBP1a conferring resistance to amoxicillin    N    609     D    S    575     T
PAIR_0013 Staphylococcus aureus rpoB mutants conferring resistance to rifampicin    A    473     T    A    477     T
PAIR_0014 Staphylococcus a

In [7]:
import json
import pandas as pd

# Define input and output paths
RAW_DIR = BASE_DIR / "raw"
PROCESSED_DIR = BASE_DIR / "processed"

JSON_PATH = RAW_DIR / "card.json"
USABLE_PATH = PROCESSED_DIR / "usable_pairs.csv"

VERIFIED_PATH = PROCESSED_DIR / "verified_pairs.csv"
EXCLUDED_PATH = PROCESSED_DIR / "excluded_pairs_step5.csv"

# Check required input files
for path in [JSON_PATH, USABLE_PATH]:
    if not path.exists():
        raise FileNotFoundError(
            f"Required input file not found: {path}"
        )

# 1. Load CARD data and map models to reference protein sequences
with JSON_PATH.open("r", encoding="utf-8") as f:
    card_data = json.load(f)

model_seq_map = {}

for model_key, entry in card_data.items():
    if not isinstance(entry, dict):
        continue

    if "model_sequences" not in entry:
        continue

    model_name = entry.get("model_name")
    aro_name = entry.get("ARO_name")
    seq_data = entry.get("model_sequences", {})

    protein_seqs = []

    if isinstance(seq_data, dict):
        for seq_key, seq_val in seq_data.items():
            if not isinstance(seq_val, dict):
                continue

            protein_info = seq_val.get("protein_sequence", {})

            if isinstance(protein_info, dict):
                p_seq = protein_info.get("sequence")

                if isinstance(p_seq, str) and p_seq.strip():
                    protein_seqs.append(p_seq.strip())

    if protein_seqs:
        if model_name:
            model_seq_map[model_name] = protein_seqs[0]

        if aro_name:
            model_seq_map[aro_name] = protein_seqs[0]

print(
    "CARD model/reference sequence identifiers mapped:",
    len(model_seq_map)
)

# 2. Load usable mutation pairs
df_usable = pd.read_csv(USABLE_PATH)

verified_pairs = []
excluded_step5 = []

# 3. Verify wild-type residues and sequence boundaries
for idx, row in df_usable.iterrows():
    pair_id = row["Pair_id"]
    model_name = row["CARD_model"]
    gene_name = row["Protein/gene"]

    wt1 = str(row["WT_1"]).strip()
    wt2 = str(row["WT_2"]).strip()

    pos1 = int(row["Pos_1"])
    pos2 = int(row["Pos_2"])

    seq = model_seq_map.get(model_name) or model_seq_map.get(gene_name)

    if not seq:
        excluded_step5.append({
            "Pair_id": pair_id,
            "Reason_excluded":
                "Reference sequence not found in CARD model_sequences",
            "CARD_model": model_name
        })
        continue

    seq_len = len(seq)

    # Check sequence boundaries
    if not (1 <= pos1 <= seq_len and 1 <= pos2 <= seq_len):
        excluded_step5.append({
            "Pair_id": pair_id,
            "Reason_excluded":
                f"Mutation position out of bounds (Seq len: {seq_len})",
            "CARD_model": model_name
        })
        continue

    # Compare reference residues
    ref_wt1 = seq[pos1 - 1]
    ref_wt2 = seq[pos2 - 1]

    if ref_wt1 != wt1 or ref_wt2 != wt2:
        mismatch_details = []

        if ref_wt1 != wt1:
            mismatch_details.append(
                f"Pos1 {wt1}->{ref_wt1}"
            )

        if ref_wt2 != wt2:
            mismatch_details.append(
                f"Pos2 {wt2}->{ref_wt2}"
            )

        excluded_step5.append({
            "Pair_id": pair_id,
            "Reason_excluded":
                "WT amino acid mismatch (" +
                "; ".join(mismatch_details) + ")",
            "CARD_model": model_name
        })
        continue

    # Retain verified pair and reference sequence
    row_dict = row.to_dict()
    row_dict["Ref_Sequence"] = seq
    row_dict["Seq_Length"] = seq_len

    verified_pairs.append(row_dict)

# 4. Create output DataFrames
df_verified = pd.DataFrame(verified_pairs)

excluded_columns = [
    "Pair_id",
    "Reason_excluded",
    "CARD_model"
]

df_excluded5 = pd.DataFrame(
    excluded_step5,
    columns=excluded_columns
)

# 5. Save outputs
df_verified.to_csv(VERIFIED_PATH, index=False)
df_excluded5.to_csv(EXCLUDED_PATH, index=False)

# 6. Report results
print(f"\nUsable pairs evaluated: {len(df_usable)}")
print(f"Verified mutation pairs retained: {len(df_verified)}")
print(f"Excluded in Step 5: {len(df_excluded5)}")

print(f"\nVerified pairs saved to: {VERIFIED_PATH}")
print(f"Excluded pairs saved to: {EXCLUDED_PATH}")

if not df_excluded5.empty:
    print("\nExclusion breakdown in Step 5:")
    print(
        df_excluded5["Reason_excluded"]
        .value_counts()
        .to_string()
    )

if not df_verified.empty:
    print("\nSample verified pairs:")

    sample_columns = [
        "Pair_id",
        "Protein/gene",
        "WT_1",
        "Pos_1",
        "WT_2",
        "Pos_2",
        "Seq_Length"
    ]

    print(
        df_verified[sample_columns]
        .head(3)
        .to_string(index=False)
    )
else:
    print("\nNo mutation pairs passed sequence verification.")

CARD model/reference sequence identifiers mapped: 0

Usable pairs evaluated: 114068
Verified mutation pairs retained: 0
Excluded in Step 5: 114068

Verified pairs saved to: /content/AMR_Structural_Mutation_Project/processed/verified_pairs.csv
Excluded pairs saved to: /content/AMR_Structural_Mutation_Project/processed/excluded_pairs_step5.csv

Exclusion breakdown in Step 5:
Reason_excluded
Reference sequence not found in CARD model_sequences    114068

No mutation pairs passed sequence verification.


In [8]:
import json

RAW_DIR = BASE_DIR / "raw"
JSON_PATH = RAW_DIR / "card.json"

if not JSON_PATH.exists():
    raise FileNotFoundError(
        f"CARD JSON file not found: {JSON_PATH}\n"
        "Please run the CARD download and extraction cell first."
    )

with JSON_PATH.open("r", encoding="utf-8") as f:
    card_data = json.load(f)

# Inspect model_sequences for three variant model entries
print("--- Inspecting CARD JSON sequence fields ---")

sample_count = 0

for model_key, entry in card_data.items():
    if (
        isinstance(entry, dict)
        and "variant" in str(entry.get("model_type", "")).lower()
    ):
        sample_count += 1

        print(f"\nVariant Model #{sample_count}:")
        print(f"  Key: {model_key}")
        print(f"  ARO_accession: {entry.get('ARO_accession')}")
        print(f"  ARO_name: {entry.get('ARO_name')}")
        print(f"  model_name: {entry.get('model_name')}")

        seqs = entry.get("model_sequences", {})
        print(f"  model_sequences type: {type(seqs).__name__}")

        if isinstance(seqs, dict):
            print(f"  model_sequences keys count: {len(seqs)}")

            first_seq_key = next(iter(seqs), None)

            if first_seq_key is not None:
                first_seq_value = seqs[first_seq_key]
                print(f"  Sample sequence key: {first_seq_key}")

                if isinstance(first_seq_value, dict):
                    print(
                        "  Sample sequence value keys: "
                        f"{list(first_seq_value.keys())}"
                    )

                    protein_info = first_seq_value.get(
                        "protein_sequence", {}
                    )

                    if isinstance(protein_info, dict):
                        sequence = protein_info.get("sequence")
                        print(
                            "  Protein sequence available: "
                            f"{isinstance(sequence, str) and bool(sequence.strip())}"
                        )
                        if isinstance(sequence, str):
                            print(f"  Protein sequence length: {len(sequence)}")
                else:
                    print(
                        "  Sample sequence value type: "
                        f"{type(first_seq_value).__name__}"
                    )

        if sample_count >= 3:
            break

if sample_count == 0:
    print("No variant model entries were found in card.json.")

print("\nCARD sequence-field inspection complete.")

--- Inspecting CARD JSON sequence fields ---

Variant Model #1:
  Key: 632
  ARO_accession: 3003582
  ARO_name: basR
  model_name: basR
  model_sequences type: dict
  model_sequences keys count: 1
  Sample sequence key: sequence
  Sample sequence value keys: ['5429']
  Protein sequence available: False

Variant Model #2:
  Key: 1607
  ARO_accession: 3003041
  ARO_name: Streptococcus pneumoniae PBP1a conferring resistance to amoxicillin
  model_name: Streptococcus pneumoniae PBP1a conferring resistance to amoxicillin
  model_sequences type: dict
  model_sequences keys count: 1
  Sample sequence key: sequence
  Sample sequence value keys: ['692']
  Protein sequence available: False

Variant Model #3:
  Key: 1966
  ARO_accession: 3003028
  ARO_name: lmrA
  model_name: lmrA
  model_sequences type: dict
  model_sequences keys count: 1
  Sample sequence key: sequence
  Sample sequence value keys: ['5186']
  Protein sequence available: False

CARD sequence-field inspection complete.


In [9]:
import json
import pandas as pd

RAW_DIR = BASE_DIR / "raw"
PROCESSED_DIR = BASE_DIR / "processed"

JSON_PATH = RAW_DIR / "card.json"
USABLE_PATH = PROCESSED_DIR / "usable_pairs.csv"
VERIFIED_PATH = PROCESSED_DIR / "verified_pairs.csv"
EXCLUDED_PATH = PROCESSED_DIR / "excluded_pairs_step5.csv"

# 1. Check required input files
for path in [JSON_PATH, USABLE_PATH]:
    if not path.exists():
        raise FileNotFoundError(
            f"Required input file not found: {path}\n"
            "Please run the preceding workflow cells first."
        )

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# 2. Load CARD JSON
with JSON_PATH.open("r", encoding="utf-8") as f:
    card_data = json.load(f)

# 3. Extract protein sequence mappings from CARD
model_seq_map = {}

for model_key, entry in card_data.items():
    if not isinstance(entry, dict):
        continue

    model_name = entry.get("model_name")
    aro_name = entry.get("ARO_name")
    model_id = entry.get("model_id")

    seqs_dict = entry.get("model_sequences", {})
    if not isinstance(seqs_dict, dict):
        continue

    # Support both common nested structures:
    # model_sequences["sequence"] = {sequence_id: {...}}
    # or model_sequences = {sequence_id: {...}}
    if "sequence" in seqs_dict and isinstance(seqs_dict["sequence"], dict):
        sequence_items = seqs_dict["sequence"].items()
    else:
        sequence_items = seqs_dict.items()

    protein_sequences = []

    for seq_id, seq_obj in sequence_items:
        if not isinstance(seq_obj, dict):
            continue

        protein_obj = seq_obj.get("protein_sequence", {})
        if not isinstance(protein_obj, dict):
            continue

        protein_sequence = protein_obj.get("sequence")

        if isinstance(protein_sequence, str) and protein_sequence.strip():
            protein_sequences.append(protein_sequence.strip())

    # Preserve the original workflow's first-sequence approach.
    # This is not sufficient to establish that the sequence is the
    # correct reference for every mutation in a CARD model.
    if protein_sequences:
        sequence = protein_sequences[0]

        for identifier in [model_name, aro_name, model_id, model_key]:
            if identifier is not None and str(identifier).strip():
                model_seq_map[str(identifier)] = sequence

print(
    "Extracted reference protein sequences for "
    f"{len(model_seq_map)} model identifiers."
)

# 4. Load usable mutation pairs
df_usable = pd.read_csv(USABLE_PATH)

required_columns = [
    "Pair_id",
    "CARD_model",
    "Protein/gene",
    "WT_1",
    "Pos_1",
    "WT_2",
    "Pos_2",
]

missing_columns = [
    column for column in required_columns
    if column not in df_usable.columns
]

if missing_columns:
    raise ValueError(
        f"Input file is missing required columns: {missing_columns}"
    )

verified_pairs = []
excluded_step5 = []

# 5. Verify reference residues and sequence boundaries
for _, row in df_usable.iterrows():
    pair_id = row["Pair_id"]
    model_name = str(row["CARD_model"]).strip()
    gene_name = str(row["Protein/gene"]).strip()

    try:
        wt1 = str(row["WT_1"]).strip()
        wt2 = str(row["WT_2"]).strip()
        pos1 = int(row["Pos_1"])
        pos2 = int(row["Pos_2"])
    except (TypeError, ValueError, OverflowError):
        excluded_step5.append({
            "Pair_id": pair_id,
            "Reason_excluded": "Invalid or missing mutation position/residue",
            "CARD_model": model_name,
        })
        continue

    sequence = model_seq_map.get(model_name)
    if not sequence:
        sequence = model_seq_map.get(gene_name)

    if not sequence:
        excluded_step5.append({
            "Pair_id": pair_id,
            "Reason_excluded": (
                "Reference sequence not found in CARD JSON "
                "for the available identifiers"
            ),
            "CARD_model": model_name,
        })
        continue

    sequence_length = len(sequence)

    if not (
        1 <= pos1 <= sequence_length
        and 1 <= pos2 <= sequence_length
    ):
        excluded_step5.append({
            "Pair_id": pair_id,
            "Reason_excluded": (
                f"Mutation position out of bounds "
                f"(sequence length: {sequence_length})"
            ),
            "CARD_model": model_name,
        })
        continue

    ref_wt1 = sequence[pos1 - 1]
    ref_wt2 = sequence[pos2 - 1]

    mismatch_details = []

    if ref_wt1 != wt1:
        mismatch_details.append(
            f"Position {pos1}: expected {wt1}, CARD sequence has {ref_wt1}"
        )

    if ref_wt2 != wt2:
        mismatch_details.append(
            f"Position {pos2}: expected {wt2}, CARD sequence has {ref_wt2}"
        )

    if mismatch_details:
        excluded_step5.append({
            "Pair_id": pair_id,
            "Reason_excluded": (
                "Wild-type amino acid mismatch: "
                + "; ".join(mismatch_details)
            ),
            "CARD_model": model_name,
        })
        continue

    row_dict = row.to_dict()
    row_dict["Ref_Sequence"] = sequence
    row_dict["Seq_Length"] = sequence_length
    verified_pairs.append(row_dict)

# 6. Save outputs with consistent columns, even if no rows pass
df_verified = pd.DataFrame(
    verified_pairs,
    columns=list(df_usable.columns) + [
        "Ref_Sequence",
        "Seq_Length",
    ],
)

df_excluded5 = pd.DataFrame(
    excluded_step5,
    columns=[
        "Pair_id",
        "Reason_excluded",
        "CARD_model",
    ],
)

df_verified.to_csv(VERIFIED_PATH, index=False)
df_excluded5.to_csv(EXCLUDED_PATH, index=False)

# 7. Report results
print(f"\nUsable pairs evaluated: {len(df_usable)}")
print(f"Verified mutation pairs retained: {len(df_verified)}")
print(f"Excluded in Step 5: {len(df_excluded5)}")

print(f"\nVerified pairs saved to: {VERIFIED_PATH}")
print(f"Excluded pairs saved to: {EXCLUDED_PATH}")

if not df_excluded5.empty:
    print("\nExclusion breakdown in Step 5:")
    print(
        df_excluded5["Reason_excluded"]
        .value_counts()
        .to_string()
    )

if not df_verified.empty:
    sample_columns = [
        "Pair_id",
        "Protein/gene",
        "WT_1",
        "Pos_1",
        "WT_2",
        "Pos_2",
        "Seq_Length",
    ]

    print("\nSample verified pairs:")
    print(
        df_verified[sample_columns]
        .head(3)
        .to_string(index=False)
    )
else:
    print(
        "\nNo mutation pairs passed sequence verification. "
        "Review the sequence mapping and exclusion reasons before "
        "continuing."
    )

Extracted reference protein sequences for 12658 model identifiers.

Usable pairs evaluated: 114068
Verified mutation pairs retained: 113398
Excluded in Step 5: 670

Verified pairs saved to: /content/AMR_Structural_Mutation_Project/processed/verified_pairs.csv
Excluded pairs saved to: /content/AMR_Structural_Mutation_Project/processed/excluded_pairs_step5.csv

Exclusion breakdown in Step 5:
Reason_excluded
Wild-type amino acid mismatch: Position 2592: expected V, CARD sequence has Y                                                    29
Wild-type amino acid mismatch: Position 2537: expected T, CARD sequence has F                                                    29
Wild-type amino acid mismatch: Position 2619: expected I, CARD sequence has S                                                    29
Wild-type amino acid mismatch: Position 2538: expected F, CARD sequence has K                                                    29
Wild-type amino acid mismatch: Position 441: expected F, CARD s

In [10]:
import json
import time
import requests
import pandas as pd

RAW_DIR = BASE_DIR / "raw"
PROCESSED_DIR = BASE_DIR / "processed"
STRUCT_DIR = RAW_DIR / "structures"

JSON_PATH = RAW_DIR / "card.json"
VERIFIED_PATH = PROCESSED_DIR / "verified_pairs.csv"
REGISTRY_PATH = PROCESSED_DIR / "structure_registry.csv"

STRUCT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# 1. Check required input files
for path in [JSON_PATH, VERIFIED_PATH]:
    if not path.exists():
        raise FileNotFoundError(
            f"Required input file not found: {path}\n"
            "Run the preceding workflow cells first."
        )

# 2. Load verified mutation pairs
df = pd.read_csv(VERIFIED_PATH)

required_columns = ["CARD_model", "Protein/gene", "Seq_Length"]
missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(
        f"verified_pairs.csv is missing required columns: {missing_columns}"
    )

unique_models = (
    df[required_columns]
    .drop_duplicates()
    .to_dict("records")
)

print(f"Total verified mutation pairs: {len(df)}")
print(f"Total unique model/gene/sequence-length combinations: {len(unique_models)}")

# 3. Load CARD JSON
with JSON_PATH.open("r", encoding="utf-8") as f:
    card_data = json.load(f)

# 4. Extract candidate UniProt accessions from CARD sequence records
# Keep the accession associated with each CARD identifier.
model_uniprot_map = {}

for model_key, entry in card_data.items():
    if not isinstance(entry, dict):
        continue

    identifiers = [
        entry.get("model_name"),
        entry.get("ARO_name"),
        entry.get("model_id"),
        model_key,
    ]

    seqs_dict = entry.get("model_sequences", {})
    if not isinstance(seqs_dict, dict):
        continue

    if "sequence" in seqs_dict and isinstance(seqs_dict["sequence"], dict):
        sequence_items = seqs_dict["sequence"].values()
    else:
        sequence_items = seqs_dict.values()

    for seq_obj in sequence_items:
        if not isinstance(seq_obj, dict):
            continue

        protein_obj = seq_obj.get("protein_sequence", {})
        if not isinstance(protein_obj, dict):
            continue

        accession = protein_obj.get("accession", "")

        if not isinstance(accession, str):
            continue

        accession = accession.strip()

        # Retain the original workflow's simple accession filter.
        # This is a candidate identifier, not proof of sequence identity.
        if accession and "_" not in accession and len(accession) in (6, 10):
            for identifier in identifiers:
                if identifier is not None and str(identifier).strip():
                    model_uniprot_map[str(identifier)] = accession

            # Keep the first eligible accession for this CARD entry.
            break

print(f"Candidate CARD-to-UniProt identifier mappings: {len(model_uniprot_map)}")

# 5. Retrieve AlphaFold DB structures
structure_registry = []
session = requests.Session()
session.headers.update({
    "User-Agent": "AMR-Structural-Epistasis-Reproducible-Research/1.0"
})

for index, item in enumerate(unique_models, start=1):
    model_name = str(item["CARD_model"]).strip()
    gene_name = str(item["Protein/gene"]).strip()
    seq_len = item["Seq_Length"]

    uniprot_acc = (
        model_uniprot_map.get(model_name)
        or model_uniprot_map.get(gene_name)
    )

    structure_id = None
    structure_source = None
    pdb_file_path = None
    status = "not_attempted"
    details = ""

    if not uniprot_acc:
        status = "no_uniprot_accession"
        details = "No candidate UniProt accession found in CARD."
    else:
        # AlphaFold DB model URL
        af_url = (
            "https://alphafold.ebi.ac.uk/files/"
            f"AF-{uniprot_acc}-F1-model_v4.pdb"
        )

        try:
            response = session.get(af_url, timeout=30)

            if response.status_code == 200 and response.text.strip():
                candidate_path = STRUCT_DIR / f"AF-{uniprot_acc}-F1.pdb"
                candidate_path.write_text(response.text, encoding="utf-8")

                # Basic file-content check; full structure validation comes later.
                if "ATOM" in response.text or "HEADER" in response.text:
                    structure_id = f"AF-{uniprot_acc}-F1"
                    structure_source = "AlphaFold DB"
                    pdb_file_path = str(candidate_path)
                    status = "downloaded"
                else:
                    candidate_path.unlink(missing_ok=True)
                    status = "invalid_response"
                    details = "Response did not appear to contain PDB records."

            elif response.status_code == 404:
                status = "structure_not_found"
                details = "AlphaFold DB returned HTTP 404."
            else:
                status = "http_error"
                details = f"AlphaFold DB returned HTTP {response.status_code}."

        except requests.RequestException as error:
            status = "request_failed"
            details = str(error)

        # Be considerate to the remote service.
        time.sleep(0.2)

    structure_registry.append({
        "CARD_model": model_name,
        "Protein/gene": gene_name,
        "Seq_Length": seq_len,
        "UniProt": uniprot_acc,
        "Structure_id": structure_id,
        "Structure_source": structure_source,
        "PDB_path": pdb_file_path,
        "Chain": "A" if structure_id else None,
        "Status": status,
        "Details": details,
    })

    if index % 25 == 0 or index == len(unique_models):
        print(f"Processed {index}/{len(unique_models)} model combinations")

session.close()

# 6. Save the structure registry
df_registry = pd.DataFrame(structure_registry)
df_registry.to_csv(REGISTRY_PATH, index=False)

mapped_count = (
    df_registry["Structure_id"].notna().sum()
    if not df_registry.empty else 0
)

print("\nStructure lookup completed.")
print(f"Model combinations with retrieved structures: {mapped_count}/{len(unique_models)}")
print(f"Structure files directory: {STRUCT_DIR}")
print(f"Structure registry saved to: {REGISTRY_PATH}")

if not df_registry.empty:
    print("\nRetrieval status summary:")
    print(df_registry["Status"].value_counts().to_string())

    print("\nSample registry entries:")
    print(df_registry.head(5).to_string(index=False))

Total verified mutation pairs: 113398
Total unique model/gene/sequence-length combinations: 129
Candidate CARD-to-UniProt identifier mappings: 8412
Processed 25/129 model combinations
Processed 50/129 model combinations
Processed 75/129 model combinations
Processed 100/129 model combinations
Processed 125/129 model combinations
Processed 129/129 model combinations

Structure lookup completed.
Model combinations with retrieved structures: 0/129
Structure files directory: /content/AMR_Structural_Mutation_Project/raw/structures
Structure registry saved to: /content/AMR_Structural_Mutation_Project/processed/structure_registry.csv

Retrieval status summary:
Status
structure_not_found     73
no_uniprot_accession    56

Sample registry entries:
                                                              CARD_model                                                             Protein/gene  Seq_Length    UniProt Structure_id Structure_source PDB_path Chain              Status                   

In [11]:
import json

RAW_DIR = BASE_DIR / "raw"
JSON_PATH = RAW_DIR / "card.json"

if not JSON_PATH.exists():
    raise FileNotFoundError(
        f"CARD JSON file not found: {JSON_PATH}\n"
        "Run the CARD download and extraction cell first."
    )

with JSON_PATH.open("r", encoding="utf-8") as f:
    card_data = json.load(f)

# Inspect sequence accessions and metadata for five sample models
sample_count = 0

print("--- Inspecting sequence metadata in card.json ---")

for model_key, entry in card_data.items():
    if not isinstance(entry, dict):
        continue

    model_type = str(entry.get("model_type", ""))

    if "variant" not in model_type.lower():
        continue

    sample_count += 1

    print(f"\nModel #{sample_count}: {entry.get('model_name')}")
    print(f"  CARD key: {model_key}")
    print(f"  ARO accession: {entry.get('ARO_accession')}")

    seqs_dict = entry.get("model_sequences", {})

    if not isinstance(seqs_dict, dict):
        print("  model_sequences is not a dictionary.")
        continue

    # Support the nested CARD structure where sequences are under "sequence".
    if (
        "sequence" in seqs_dict
        and isinstance(seqs_dict["sequence"], dict)
    ):
        sequence_items = list(seqs_dict["sequence"].items())
    else:
        sequence_items = list(seqs_dict.items())

    if not sequence_items:
        print("  No sequence records found.")
    else:
        for seq_id, seq_obj in sequence_items[:2]:
            print(f"  Sequence ID: {seq_id}")

            if not isinstance(seq_obj, dict):
                print(f"    Sequence record type: {type(seq_obj).__name__}")
                continue

            protein_obj = seq_obj.get("protein_sequence", {})

            if not isinstance(protein_obj, dict):
                print("    protein_sequence metadata is unavailable.")
                continue

            print(f"    Protein sequence keys: {list(protein_obj.keys())}")
            print(f"    Accession: {protein_obj.get('accession')}")
            print(f"    Description: {protein_obj.get('description')}")

            sequence = protein_obj.get("sequence")
            if isinstance(sequence, str):
                print(f"    Sequence length: {len(sequence)}")

    if sample_count >= 5:
        break

if sample_count == 0:
    print("No CARD variant model entries were found.")

print("\nSequence metadata inspection complete.")

--- Inspecting sequence metadata in card.json ---

Model #1: basR
  CARD key: 632
  ARO accession: 3003582
  Sequence ID: 5429
    Protein sequence keys: ['accession', 'sequence']
    Accession: AAG08162.1
    Description: None
    Sequence length: 221

Model #2: Streptococcus pneumoniae PBP1a conferring resistance to amoxicillin
  CARD key: 1607
  ARO accession: 3003041
  Sequence ID: 692
    Protein sequence keys: ['accession', 'sequence']
    Accession: AFC91828.1
    Description: None
    Sequence length: 719

Model #3: lmrA
  CARD key: 1966
  ARO accession: 3003028
  Sequence ID: 5186
    Protein sequence keys: ['accession', 'sequence']
    Accession: CAA42550.1
    Description: None
    Sequence length: 481

Model #4: Escherichia coli folP with mutation conferring resistance to sulfonamides
  CARD key: 29
  ARO accession: 3003386
  Sequence ID: 5412
    Protein sequence keys: ['accession', 'sequence']
    Accession: AAC76209.2
    Description: None
    Sequence length: 282

Model

In [12]:
import json
import time
import requests
import pandas as pd

RAW_DIR = BASE_DIR / "raw"
PROCESSED_DIR = BASE_DIR / "processed"
STRUCT_DIR = RAW_DIR / "structures"

JSON_PATH = RAW_DIR / "card.json"
VERIFIED_PATH = PROCESSED_DIR / "verified_pairs.csv"
MAPPING_PATH = PROCESSED_DIR / "structure_mapping.csv"

STRUCT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# 1. Validate and load input files
for path in [JSON_PATH, VERIFIED_PATH]:
    if not path.exists():
        raise FileNotFoundError(
            f"Required input file not found: {path}\n"
            "Run the preceding workflow cells first."
        )

df = pd.read_csv(VERIFIED_PATH)

required_columns = ["CARD_model", "Protein/gene", "Seq_Length"]
missing_columns = [
    col for col in required_columns if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"verified_pairs.csv is missing required columns: {missing_columns}"
    )

with JSON_PATH.open("r", encoding="utf-8") as f:
    card_data = json.load(f)

# 2. Extract candidate NCBI protein accessions from CARD
model_ncbi_map = {}

for model_key, entry in card_data.items():
    if not isinstance(entry, dict):
        continue

    identifiers = [
        entry.get("model_name"),
        entry.get("ARO_name"),
        entry.get("model_id"),
        model_key,
    ]

    seqs_dict = entry.get("model_sequences", {})
    if not isinstance(seqs_dict, dict):
        continue

    if (
        "sequence" in seqs_dict
        and isinstance(seqs_dict["sequence"], dict)
    ):
        sequence_items = seqs_dict["sequence"].values()
    else:
        sequence_items = seqs_dict.values()

    for seq_obj in sequence_items:
        if not isinstance(seq_obj, dict):
            continue

        protein_obj = seq_obj.get("protein_sequence", {})
        if not isinstance(protein_obj, dict):
            continue

        accession = protein_obj.get("accession")

        if isinstance(accession, str) and accession.strip():
            accession = accession.strip()

            # Remove a version suffix only for accession formats
            # that use a numeric suffix, e.g. WP_012345678.1.
            if "." in accession:
                base, suffix = accession.rsplit(".", 1)
                if suffix.isdigit():
                    accession = base

            for identifier in identifiers:
                if identifier is not None and str(identifier).strip():
                    model_ncbi_map[str(identifier)] = accession

            # Preserve the original workflow's first-accession approach.
            break

print(
    "CARD identifiers with candidate protein accessions:",
    len(model_ncbi_map),
)

# 3. Extract unique model/gene/sequence-length combinations
unique_models = (
    df[required_columns]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(f"Total verified mutation pairs: {len(df)}")
print(f"Total unique model combinations: {len(unique_models)}")

# 4. Map candidate NCBI accessions to UniProt and retrieve structures
model_structure_info = []

session = requests.Session()
session.headers.update({
    "User-Agent": "AMR-Structural-Epistasis-Reproducible-Research/1.0"
})

print("\nMapping protein accessions and retrieving AlphaFold structures...")

for index, row in unique_models.iterrows():
    model_name = str(row["CARD_model"]).strip()
    gene_name = str(row["Protein/gene"]).strip()
    seq_len = row["Seq_Length"]

    ncbi_acc = (
        model_ncbi_map.get(model_name)
        or model_ncbi_map.get(gene_name)
    )

    uniprot_acc = None
    structure_id = None
    structure_source = None
    pdb_path = None
    status = "not_attempted"
    details = ""

    # 4a. Query UniProt for a candidate mapping
    if not ncbi_acc:
        status = "no_ncbi_accession"
        details = "No candidate protein accession found in CARD."
    else:
        try:
            uniprot_url = "https://rest.uniprot.org/uniprotkb/search"
            response = session.get(
                uniprot_url,
                params={
                    "query": f"accession:{ncbi_acc}",
                    "format": "json",
                    "size": 5,
                },
                timeout=20,
            )

            if response.status_code == 200:
                results = response.json().get("results", [])

                if results:
                    # Use the first result as a candidate only.
                    # Sequence-level validation is still required.
                    uniprot_acc = results[0].get("primaryAccession")
                else:
                    status = "uniprot_mapping_not_found"
                    details = "UniProt returned no matching records."

            else:
                status = "uniprot_http_error"
                details = f"UniProt returned HTTP {response.status_code}."

        except (requests.RequestException, ValueError) as error:
            status = "uniprot_request_failed"
            details = str(error)

    # 4b. Download candidate AlphaFold structure
    if uniprot_acc:
        af_url = (
            "https://alphafold.ebi.ac.uk/files/"
            f"AF-{uniprot_acc}-F1-model_v4.pdb"
        )

        try:
            af_response = session.get(af_url, timeout=30)

            if (
                af_response.status_code == 200
                and (
                    "ATOM" in af_response.text
                    or "HEADER" in af_response.text
                )
            ):
                candidate_path = STRUCT_DIR / f"AF-{uniprot_acc}-F1.pdb"
                candidate_path.write_text(
                    af_response.text,
                    encoding="utf-8",
                )

                structure_id = f"AF-{uniprot_acc}-F1"
                structure_source = "AlphaFold DB"
                pdb_path = str(candidate_path)
                status = "downloaded"
                details = (
                    "Candidate structure downloaded; sequence identity "
                    "and residue numbering still require validation."
                )

            elif af_response.status_code == 404:
                status = "structure_not_found"
                details = "AlphaFold DB returned HTTP 404."

            else:
                status = "structure_download_failed"
                details = (
                    f"AlphaFold DB returned HTTP {af_response.status_code} "
                    "or the response was not recognized as PDB content."
                )

        except requests.RequestException as error:
            status = "structure_request_failed"
            details = str(error)

    model_structure_info.append({
        "CARD_model": model_name,
        "Protein/gene": gene_name,
        "Seq_Length": seq_len,
        "NCBI_Acc": ncbi_acc,
        "UniProt_Acc": uniprot_acc,
        "Structure_id": structure_id,
        "Structure_source": structure_source,
        "PDB_path": pdb_path,
        "Chain": "A" if structure_id else None,
        "Status": status,
        "Details": details,
    })

    if (index + 1) % 25 == 0 or index + 1 == len(unique_models):
        print(f"Processed {index + 1}/{len(unique_models)} model combinations")

    time.sleep(0.2)

session.close()

# 5. Save structure mapping table
df_struct_map = pd.DataFrame(model_structure_info)
df_struct_map.to_csv(MAPPING_PATH, index=False)

mapped_models = (
    df_struct_map["Structure_id"].notna().sum()
    if not df_struct_map.empty else 0
)

print("\nStructure lookup complete.")
print(
    "Unique model combinations with downloaded structures: "
    f"{mapped_models}/{len(unique_models)}"
)
print(f"Structure files directory: {STRUCT_DIR}")
print(f"Mapping table saved to: {MAPPING_PATH}")

if not df_struct_map.empty:
    print("\nStatus summary:")
    print(df_struct_map["Status"].value_counts().to_string())

CARD identifiers with candidate protein accessions: 12658
Total verified mutation pairs: 113398
Total unique model combinations: 129

Mapping protein accessions and retrieving AlphaFold structures...
Processed 25/129 model combinations
Processed 50/129 model combinations
Processed 75/129 model combinations
Processed 100/129 model combinations
Processed 125/129 model combinations
Processed 129/129 model combinations

Structure lookup complete.
Unique model combinations with downloaded structures: 0/129
Structure files directory: /content/AMR_Structural_Mutation_Project/raw/structures
Mapping table saved to: /content/AMR_Structural_Mutation_Project/processed/structure_mapping.csv

Status summary:
Status
uniprot_http_error    129


In [13]:
import requests

test_accessions = [
    "AAG08162.1",
    "AAG08162",
    "AAC76209.2",
    "AAC76209",
    "AFC91828.1",
]

session = requests.Session()
session.headers.update({
    "User-Agent": "AMR-Structural-Epistasis-Reproducible-Research/1.0"
})

print("--- Testing UniProt API search syntax ---")

for accession in test_accessions:
    results_by_query = {}

    queries = {
        "General Query": accession,
        "Accession Query": f"accession:{accession}",
        "xref Query": f"xref:{accession}",
    }

    for label, query in queries.items():
        try:
            response = session.get(
                "https://rest.uniprot.org/uniprotkb/search",
                params={
                    "query": query,
                    "format": "json",
                    "size": 1,
                },
                timeout=20,
            )
            response.raise_for_status()

            results = response.json().get("results", [])

            if results:
                results_by_query[label] = (
                    results[0].get("primaryAccession", "Unknown")
                )
            else:
                results_by_query[label] = "None"

        except (requests.RequestException, ValueError) as error:
            results_by_query[label] = f"Error: {error}"

    print(
        f"Accession: {accession:12s} | "
        f"General: {results_by_query['General Query']:20s} | "
        f"Accession field: {results_by_query['Accession Query']:20s} | "
        f"xref: {results_by_query['xref Query']}"
    )

session.close()

print("\nUniProt API syntax test complete.")

--- Testing UniProt API search syntax ---
Accession: AAG08162.1   | General: Q9HV32               | Accession field: Error: 400 Client Error: Bad Request for url: https://rest.uniprot.org/uniprotkb/search?query=accession%3AAAG08162.1&format=json&size=1 | xref: Q9HV32
Accession: AAG08162     | General: Q9HV32               | Accession field: Error: 400 Client Error: Bad Request for url: https://rest.uniprot.org/uniprotkb/search?query=accession%3AAAG08162&format=json&size=1 | xref: Q9HV32
Accession: AAC76209.2   | General: P0AC13               | Accession field: Error: 400 Client Error: Bad Request for url: https://rest.uniprot.org/uniprotkb/search?query=accession%3AAAC76209.2&format=json&size=1 | xref: P0AC13
Accession: AAC76209     | General: P0AC13               | Accession field: Error: 400 Client Error: Bad Request for url: https://rest.uniprot.org/uniprotkb/search?query=accession%3AAAC76209&format=json&size=1 | xref: P0AC13
Accession: AFC91828.1   | General: H8ZYN4               | 

In [14]:
import json
import time
import requests
import pandas as pd

RAW_DIR = BASE_DIR / "raw"
PROCESSED_DIR = BASE_DIR / "processed"
STRUCT_DIR = RAW_DIR / "structures"

JSON_PATH = RAW_DIR / "card.json"
VERIFIED_PATH = PROCESSED_DIR / "verified_pairs.csv"
MAPPING_PATH = PROCESSED_DIR / "structure_mapping.csv"

STRUCT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# 1. Check input files
for path in [JSON_PATH, VERIFIED_PATH]:
    if not path.exists():
        raise FileNotFoundError(
            f"Required input file not found: {path}\n"
            "Run the preceding workflow cells first."
        )

df_verified = pd.read_csv(VERIFIED_PATH)

required_columns = ["CARD_model", "Protein/gene", "Seq_Length"]
missing_columns = [
    column for column in required_columns
    if column not in df_verified.columns
]

if missing_columns:
    raise ValueError(
        f"verified_pairs.csv is missing required columns: {missing_columns}"
    )

# 2. Load CARD JSON
with JSON_PATH.open("r", encoding="utf-8") as f:
    card_data = json.load(f)

# 3. Extract candidate protein accessions from CARD
model_ncbi_map = {}

for model_key, entry in card_data.items():
    if not isinstance(entry, dict):
        continue

    identifiers = [
        entry.get("model_name"),
        entry.get("ARO_name"),
        entry.get("model_id"),
        model_key,
    ]

    seqs_dict = entry.get("model_sequences", {})
    if not isinstance(seqs_dict, dict):
        continue

    # Support nested sequence structures.
    if (
        "sequence" in seqs_dict
        and isinstance(seqs_dict["sequence"], dict)
    ):
        sequence_items = seqs_dict["sequence"].values()
    else:
        sequence_items = seqs_dict.values()

    for seq_obj in sequence_items:
        if not isinstance(seq_obj, dict):
            continue

        protein_obj = seq_obj.get("protein_sequence", {})
        if not isinstance(protein_obj, dict):
            continue

        accession = protein_obj.get("accession")

        if isinstance(accession, str) and accession.strip():
            accession = accession.strip()

            for identifier in identifiers:
                if identifier is not None and str(identifier).strip():
                    model_ncbi_map[str(identifier)] = accession

            # Preserve the original workflow's first-accession approach.
            break

# 4. Identify unique protein model combinations
unique_models = (
    df_verified[required_columns]
    .drop_duplicates()
    .to_dict("records")
)

print(f"Total verified mutation pairs: {len(df_verified)}")
print(f"Total unique model combinations: {len(unique_models)}")

# 5. Map candidate accessions and retrieve AlphaFold structures
model_struct_records = []

session = requests.Session()
session.headers.update({
    "User-Agent": "AMR-Structural-Epistasis-Reproducible-Research/1.0"
})

for index, item in enumerate(unique_models, start=1):
    model_name = str(item["CARD_model"]).strip()
    gene_name = str(item["Protein/gene"]).strip()
    seq_len = item["Seq_Length"]

    ncbi_acc = (
        model_ncbi_map.get(model_name)
        or model_ncbi_map.get(gene_name)
    )

    uniprot_acc = None
    structure_id = None
    structure_source = None
    pdb_path = None
    status = "not_attempted"
    details = ""

    # 5a. Search UniProt using the candidate cross-reference
    if not ncbi_acc:
        status = "no_protein_accession"
        details = "No protein accession was found in CARD."
    else:
        try:
            response = session.get(
                "https://rest.uniprot.org/uniprotkb/search",
                params={
                    "query": f"xref:{ncbi_acc}",
                    "format": "json",
                    "size": 5,
                },
                timeout=20,
            )
            response.raise_for_status()

            results = response.json().get("results", [])

            if results:
                # Candidate only; validate its sequence before analysis.
                uniprot_acc = results[0].get("primaryAccession")
            else:
                status = "uniprot_mapping_not_found"
                details = "No UniProt cross-reference result found."

        except (requests.RequestException, ValueError) as error:
            status = "uniprot_request_failed"
            details = str(error)

    # 5b. Query AlphaFold DB for the candidate UniProt accession
    if uniprot_acc:
        try:
            api_url = (
                "https://alphafold.ebi.ac.uk/api/prediction/"
                f"{uniprot_acc}"
            )

            api_response = session.get(api_url, timeout=20)
            api_response.raise_for_status()
            api_data = api_response.json()

            if not isinstance(api_data, list) or not api_data:
                status = "alphafold_entry_not_found"
                details = "AlphaFold API returned no prediction records."
            else:
                # Prefer the first prediction returned by the API.
                prediction = api_data[0]
                pdb_url = prediction.get("pdbUrl")

                if not pdb_url:
                    status = "pdb_url_missing"
                    details = "AlphaFold API record did not include pdbUrl."
                else:
                    pdb_response = session.get(pdb_url, timeout=30)
                    pdb_response.raise_for_status()

                    pdb_text = pdb_response.text

                    # Basic validation that the response resembles a PDB file.
                    if not any(
                        line.startswith(("ATOM  ", "HETATM", "HEADER"))
                        for line in pdb_text.splitlines()
                    ):
                        status = "invalid_pdb_response"
                        details = "Downloaded response did not resemble a PDB file."
                    else:
                        api_structure_id = (
                            prediction.get("entryId")
                            or f"AF-{uniprot_acc}"
                        )
                        structure_id = str(api_structure_id)

                        # Use a filesystem-safe filename.
                        safe_id = "".join(
                            char if char.isalnum() or char in "-_."
                            else "_"
                            for char in structure_id
                        )

                        output_path = STRUCT_DIR / f"{safe_id}.pdb"
                        output_path.write_text(
                            pdb_text,
                            encoding="utf-8",
                        )

                        structure_source = "AlphaFold DB"
                        pdb_path = str(output_path)
                        status = "downloaded"
                        details = (
                            "PDB downloaded; sequence identity, chain, "
                            "and residue numbering still require validation."
                        )

        except (requests.RequestException, ValueError) as error:
            status = "alphafold_request_failed"
            details = str(error)

    model_struct_records.append({
        "CARD_model": model_name,
        "Protein/gene": gene_name,
        "Seq_Length": seq_len,
        "NCBI_Acc": ncbi_acc,
        "UniProt_Acc": uniprot_acc,
        "Structure_id": structure_id,
        "Structure_source": structure_source,
        "PDB_path": pdb_path,
        "Chain": "A" if pdb_path else None,
        "Status": status,
        "Details": details,
    })

    if index % 25 == 0 or index == len(unique_models):
        print(f"Processed {index}/{len(unique_models)} model combinations")

    time.sleep(0.2)

session.close()

# 6. Save the structure mapping table
df_struct = pd.DataFrame(model_struct_records)
df_struct.to_csv(MAPPING_PATH, index=False)

mapped_count = (
    df_struct["PDB_path"].apply(
        lambda path: isinstance(path, str) and (BASE_DIR / "raw" / "structures" / Path(path).name).exists()
    ).sum()
    if not df_struct.empty else 0
)

print("\nStructure retrieval completed.")
print(
    f"Successfully retrieved PDB files: "
    f"{mapped_count}/{len(unique_models)}"
)
print(f"Structure directory: {STRUCT_DIR}")
print(f"Mapping table saved to: {MAPPING_PATH}")

if not df_struct.empty:
    print("\nRetrieval status summary:")
    print(df_struct["Status"].value_counts().to_string())

Total verified mutation pairs: 113398
Total unique model combinations: 129
Processed 25/129 model combinations
Processed 50/129 model combinations
Processed 75/129 model combinations
Processed 100/129 model combinations
Processed 125/129 model combinations
Processed 129/129 model combinations

Structure retrieval completed.
Successfully retrieved PDB files: 113/129
Structure directory: /content/AMR_Structural_Mutation_Project/raw/structures
Mapping table saved to: /content/AMR_Structural_Mutation_Project/processed/structure_mapping.csv

Retrieval status summary:
Status
downloaded                   113
uniprot_mapping_not_found     14
alphafold_request_failed       2


In [15]:
import math
import pandas as pd
from pathlib import Path

PROCESSED_DIR = BASE_DIR / "processed"
STRUCT_DIR = BASE_DIR / "raw" / "structures"

PAIRS_PATH = PROCESSED_DIR / "verified_pairs.csv"
STRUCTURE_MAP_PATH = PROCESSED_DIR / "structure_mapping.csv"

DISTANCE_PATH = PROCESSED_DIR / "struct_distance_pairs.csv"
EXCLUDED_PATH = PROCESSED_DIR / "excluded_pairs_step6.csv"

# 1. Validate input files
for path in [PAIRS_PATH, STRUCTURE_MAP_PATH]:
    if not path.exists():
        raise FileNotFoundError(
            f"Required input file not found: {path}\n"
            "Run the preceding workflow cells first."
        )

df_pairs = pd.read_csv(PAIRS_PATH)
df_struct = pd.read_csv(STRUCTURE_MAP_PATH)

required_pair_columns = [
    "Pair_id", "CARD_model", "Pos_1", "Pos_2"
]
required_structure_columns = [
    "CARD_model", "Structure_id", "Structure_source",
    "PDB_path", "Chain"
]

missing_pair_columns = [
    col for col in required_pair_columns if col not in df_pairs.columns
]
missing_structure_columns = [
    col for col in required_structure_columns if col not in df_struct.columns
]

if missing_pair_columns:
    raise ValueError(f"Missing pair columns: {missing_pair_columns}")

if missing_structure_columns:
    raise ValueError(
        f"Missing structure mapping columns: {missing_structure_columns}"
    )

# 2. Build a structure metadata lookup
# The mapping table should contain one entry per CARD_model.
# If duplicates exist, keep the first non-null structure record.
df_struct = df_struct.drop_duplicates(
    subset=["CARD_model"],
    keep="first",
)
struct_map = df_struct.set_index("CARD_model").to_dict("index")


def load_ca_coords(pdb_path, chain_id="A"):
    """
    Parse C-alpha coordinates from a PDB file for a specific chain.

    Returns:
        Dictionary mapping (residue_number, insertion_code) to
        (x, y, z) coordinates.

    Note:
        This parser uses PDB author residue numbering. It does not
        automatically map CARD sequence positions to PDB residue IDs.
    """
    coords = {}

    if not isinstance(pdb_path, str) or not pdb_path.strip():
        return coords

    path = Path(pdb_path)

    # Handle paths stored as absolute paths or project-relative paths.
    if not path.exists():
        candidate = STRUCT_DIR / path.name
        if candidate.exists():
            path = candidate
        else:
            return coords

    with path.open("r", encoding="utf-8", errors="replace") as handle:
        for line in handle:
            if not line.startswith("ATOM  "):
                continue

            atom_name = line[12:16].strip()
            if atom_name != "CA":
                continue

            current_chain = line[21].strip() or " "
            requested_chain = str(chain_id).strip() or " "

            if current_chain != requested_chain:
                continue

            try:
                residue_number = int(line[22:26].strip())
                insertion_code = line[26].strip()
                x = float(line[30:38].strip())
                y = float(line[38:46].strip())
                z = float(line[46:54].strip())
            except (ValueError, IndexError):
                continue

            residue_key = (residue_number, insertion_code)

            # Keep the first CA record for each residue.
            if residue_key not in coords:
                coords[residue_key] = (x, y, z)

    return coords


# 3. Cache parsed coordinates by file and chain
ca_cache = {}

retained_3d_pairs = []
excluded_step6 = []

for _, row in df_pairs.iterrows():
    pair_id = row["Pair_id"]
    model_name = row["CARD_model"]

    try:
        pos1 = int(row["Pos_1"])
        pos2 = int(row["Pos_2"])
    except (TypeError, ValueError, OverflowError):
        excluded_step6.append({
            "Pair_id": pair_id,
            "Reason_excluded": "Invalid or missing mutation position",
            "CARD_model": model_name,
        })
        continue

    structure_info = struct_map.get(model_name, {})

    structure_id = structure_info.get("Structure_id")
    structure_source = structure_info.get("Structure_source")
    pdb_path = structure_info.get("PDB_path")
    chain = structure_info.get("Chain", "A")

    if pd.isna(structure_id) or not structure_id:
        excluded_step6.append({
            "Pair_id": pair_id,
            "Reason_excluded": "3D structure unavailable for protein model",
            "CARD_model": model_name,
        })
        continue

    if not isinstance(pdb_path, str) or not pdb_path.strip() or pd.isna(pdb_path):
        excluded_step6.append({
            "Pair_id": pair_id,
            "Reason_excluded": "PDB file path missing from structure mapping",
            "CARD_model": model_name,
        })
        continue

    # Normalize the path for reliable cache keys.
    resolved_path = Path(pdb_path)
    if not resolved_path.exists():
        resolved_path = STRUCT_DIR / resolved_path.name

    if not resolved_path.exists():
        excluded_step6.append({
            "Pair_id": pair_id,
            "Reason_excluded": "PDB file not found on disk",
            "CARD_model": model_name,
        })
        continue

    chain = "A" if pd.isna(chain) or not str(chain).strip() else str(chain).strip()
    cache_key = (str(resolved_path.resolve()), chain)

    if cache_key not in ca_cache:
        ca_cache[cache_key] = load_ca_coords(
            str(resolved_path),
            chain_id=chain,
        )

    coords = ca_cache[cache_key]

    # This direct lookup assumes sequence positions equal PDB residue numbers
    # and that residues have no insertion-code ambiguity.
    coord1 = coords.get((pos1, ""))
    coord2 = coords.get((pos2, ""))

    if coord1 is None or coord2 is None:
        missing = []
        if coord1 is None:
            missing.append(f"position {pos1}")
        if coord2 is None:
            missing.append(f"position {pos2}")

        excluded_step6.append({
            "Pair_id": pair_id,
            "Reason_excluded": (
                "C-alpha coordinate missing for "
                + ", ".join(missing)
                + f" in chain {chain}"
            ),
            "CARD_model": model_name,
        })
        continue

    # 4. Calculate sequence separation and Euclidean C-alpha distance
    sequence_distance = abs(pos1 - pos2)

    distance_3d = math.sqrt(
        (coord1[0] - coord2[0]) ** 2
        + (coord1[1] - coord2[1]) ** 2
        + (coord1[2] - coord2[2]) ** 2
    )

    result = row.to_dict()
    result["Structure_id"] = structure_id
    result["Structure_source"] = structure_source
    result["Chain"] = chain
    result["Sequence_distance"] = sequence_distance
    result["3D_distance"] = round(distance_3d, 3)
    result["Distance_unit"] = "angstrom"
    result["Residue_mapping_method"] = "direct_position_to_PDB_number"

    retained_3d_pairs.append(result)

# 5. Save outputs, preserving columns if no rows pass
df_3d = pd.DataFrame(retained_3d_pairs)

df_ex6 = pd.DataFrame(
    excluded_step6,
    columns=["Pair_id", "Reason_excluded", "CARD_model"],
)

df_3d.to_csv(DISTANCE_PATH, index=False)
df_ex6.to_csv(EXCLUDED_PATH, index=False)

# 6. Report results
print(f"Verified mutation pairs evaluated: {len(df_pairs)}")
print(f"Pairs with calculated C-alpha distances: {len(df_3d)}")
print(f"Pairs excluded in Step 6: {len(df_ex6)}")
print(f"\nDistance table saved to: {DISTANCE_PATH}")
print(f"Exclusion report saved to: {EXCLUDED_PATH}")

if not df_ex6.empty:
    print("\nExclusion breakdown:")
    print(df_ex6["Reason_excluded"].value_counts().to_string())

if not df_3d.empty:
    sample_columns = [
        "Pair_id",
        "Protein/gene",
        "WT_1",
        "Pos_1",
        "WT_2",
        "Pos_2",
        "Sequence_distance",
        "3D_distance",
        "Structure_id",
    ]

    print("\nSample calculated distances:")
    print(df_3d[sample_columns].head(5).to_string(index=False))
else:
    print(
        "\nNo pairs had both C-alpha coordinates available. "
        "Review the structure mapping, chain IDs, and residue numbering."
    )

Verified mutation pairs evaluated: 113398
Pairs with calculated C-alpha distances: 112882
Pairs excluded in Step 6: 516

Distance table saved to: /content/AMR_Structural_Mutation_Project/processed/struct_distance_pairs.csv
Exclusion report saved to: /content/AMR_Structural_Mutation_Project/processed/excluded_pairs_step6.csv

Exclusion breakdown:
Reason_excluded
3D structure unavailable for protein model    516

Sample calculated distances:
  Pair_id                                                           Protein/gene WT_1  Pos_1 WT_2  Pos_2  Sequence_distance  3D_distance Structure_id
PAIR_0001                                                                   basR    L    157    L     71                 86       24.773 AF-Q9HV32-F1
PAIR_0002    Streptococcus pneumoniae PBP1a conferring resistance to amoxicillin    N    609    S    575                 34       16.859 AF-H8ZYN4-F1
PAIR_0013 Staphylococcus aureus rpoB mutants conferring resistance to rifampicin    A    473    A    477  

In [16]:
import math
import random
import pandas as pd
from pathlib import Path

# --------------------------------------------------
# 1. Define paths and load input files
# --------------------------------------------------
PROCESSED_DIR = BASE_DIR / "processed"
STRUCT_DIR = BASE_DIR / "raw" / "structures"

DISTANCE_PATH = PROCESSED_DIR / "struct_distance_pairs.csv"
STRUCTURE_MAP_PATH = PROCESSED_DIR / "structure_mapping.csv"
CONTROL_PATH = PROCESSED_DIR / "matched_controls.csv"

df_obs = pd.read_csv(DISTANCE_PATH)
df_struct = pd.read_csv(STRUCTURE_MAP_PATH)

required_obs_cols = {"Pair_id", "Structure_id", "Sequence_distance"}
required_map_cols = {"CARD_model", "Structure_id", "PDB_path"}

missing_obs = required_obs_cols - set(df_obs.columns)
missing_map = required_map_cols - set(df_struct.columns)

if missing_obs:
    raise ValueError(f"Missing columns in struct_distance_pairs.csv: {sorted(missing_obs)}")
if missing_map:
    raise ValueError(f"Missing columns in structure_mapping.csv: {sorted(missing_map)}")

# One mapping record per structure ID. Keep only usable rows.
df_struct = df_struct.dropna(subset=["Structure_id"]).copy()
struct_map = (
    df_struct.drop_duplicates(subset=["Structure_id"])
    .set_index("Structure_id")
    .to_dict("index")
)

# --------------------------------------------------
# 2. Parse C-alpha coordinates
#    Keep chain and insertion code to avoid residue-key
#    collisions in multi-chain or insertion-coded PDBs.
# --------------------------------------------------
def resolve_pdb_path(pdb_path):
    """Resolve a recorded PDB path, falling back to the structures directory."""
    if not isinstance(pdb_path, str) or not pdb_path.strip():
        return None

    candidate = Path(pdb_path)

    if candidate.exists():
        return candidate

    fallback = STRUCT_DIR / candidate.name
    if fallback.exists():
        return fallback

    return None


def load_ca_coords(pdb_path, chain_id="A"):
    """
    Return {(residue_number, insertion_code): (x, y, z)}
    for C-alpha atoms in the specified chain.
    """
    coords = {}
    resolved_path = resolve_pdb_path(str(pdb_path) if pdb_path is not None else "")

    if resolved_path is None:
        return coords

    with resolved_path.open("r", encoding="utf-8", errors="replace") as f:
        for line in f:
            if not line.startswith("ATOM  "):
                continue

            atom_name = line[12:16].strip()
            atom_altloc = line[16:17].strip()
            atom_chain = line[21:22].strip() or " "

            if atom_name != "CA":
                continue
            if atom_chain != (str(chain_id).strip() or " "):
                continue
            # Prefer the primary location; accept blank or A alternate locations.
            if atom_altloc not in ("", "A"):
                continue

            try:
                res_num = int(line[22:26].strip())
                insertion_code = line[26:27].strip()
                x = float(line[30:38])
                y = float(line[38:46])
                z = float(line[46:54])
            except (ValueError, IndexError):
                continue

            key = (res_num, insertion_code)
            if key not in coords:
                coords[key] = (x, y, z)

    return coords


def residue_number_distance(res1, res2):
    """Calculate absolute residue-number difference."""
    return abs(res1[0] - res2[0])


def euclidean_distance(c1, c2):
    """Calculate C-alpha distance in angstroms."""
    return math.sqrt(
        (c1[0] - c2[0]) ** 2
        + (c1[1] - c2[1]) ** 2
        + (c1[2] - c2[2]) ** 2
    )


# --------------------------------------------------
# 3. Cache coordinates and index possible control pairs
# --------------------------------------------------
print("Caching structure coordinates and indexing residue pairs...")

ca_cache = {}
struct_seqdist_index = {}
structure_path_cache = {}

for structure_id, s_info in struct_map.items():
    pdb_path = s_info.get("PDB_path")
    chain_id = s_info.get("Chain", "A")

    resolved_path = resolve_pdb_path(pdb_path)
    if resolved_path is None:
        continue

    coords = load_ca_coords(resolved_path, chain_id=chain_id)

    if len(coords) < 2:
        continue

    ca_cache[structure_id] = coords
    structure_path_cache[structure_id] = str(resolved_path)

    # Use residue-number separation as a proxy for sequence separation.
    # This is reliable only when PDB numbering corresponds to sequence positions.
    residue_keys = sorted(coords.keys())
    distance_index = {}

    for i in range(len(residue_keys)):
        for j in range(i + 1, len(residue_keys)):
            r1 = residue_keys[i]
            r2 = residue_keys[j]
            seq_sep = residue_number_distance(r1, r2)
            distance_index.setdefault(seq_sep, []).append((r1, r2))

    struct_seqdist_index[structure_id] = distance_index

print(f"Structures with usable C-alpha coordinates: {len(ca_cache)}")

# --------------------------------------------------
# 4. Generate sequence-distance-matched random controls
# --------------------------------------------------
rng = random.Random(42)
control_records = []

for _, row in df_obs.iterrows():
    pair_id = row["Pair_id"]
    structure_id = row["Structure_id"]

    if pd.isna(structure_id) or pd.isna(row["Sequence_distance"]):
        continue

    try:
        observed_seq_dist = int(row["Sequence_distance"])
    except (TypeError, ValueError, OverflowError):
        continue

    coords = ca_cache.get(structure_id)
    distance_index = struct_seqdist_index.get(structure_id)

    if not coords or not distance_index:
        continue

    candidates = distance_index.get(observed_seq_dist, [])
    match_type = "exact"

    if not candidates:
        available_distances = list(distance_index.keys())
        if not available_distances:
            continue

        nearest_distance = min(
            available_distances,
            key=lambda d: (abs(d - observed_seq_dist), d)
        )
        candidates = distance_index[nearest_distance]
        match_type = "nearest_fallback"

    if not candidates:
        continue

    control_pos_1, control_pos_2 = rng.choice(candidates)
    coord_1 = coords[control_pos_1]
    coord_2 = coords[control_pos_2]

    control_3d_dist = euclidean_distance(coord_1, coord_2)
    control_seq_dist = residue_number_distance(
        control_pos_1, control_pos_2
    )

    control_records.append({
        "Pair_id": pair_id,
        "Structure_id": structure_id,
        "Control_Pos_1": control_pos_1[0],
        "Control_Insertion_Code_1": control_pos_1[1],
        "Control_Pos_2": control_pos_2[0],
        "Control_Insertion_Code_2": control_pos_2[1],
        "Control_Sequence_distance": control_seq_dist,
        "Observed_Sequence_distance": observed_seq_dist,
        "Control_3D_distance": round(control_3d_dist, 3),
        "Distance_unit": "angstrom",
        "Match_type": match_type,
        "PDB_path": structure_path_cache.get(structure_id)
    })

df_ctrl = pd.DataFrame(control_records)
df_ctrl.to_csv(CONTROL_PATH, index=False)

# --------------------------------------------------
# 5. Report results and matching quality
# --------------------------------------------------
print(f"\nGenerated controls for {len(df_ctrl)} of {len(df_obs)} observed pairs.")
print(f"Saved to: {CONTROL_PATH}")

if not df_ctrl.empty:
    print("\nMatch-type summary:")
    print(df_ctrl["Match_type"].value_counts(dropna=False).to_string())

    print("\nSample matched controls:")
    print(df_ctrl.head(5).to_string(index=False))
else:
    print(
        "No controls were generated. Check Structure_id values, PDB paths, "
        "chain IDs, and whether the structures contain C-alpha atoms."
    )

Caching structure coordinates and indexing residue pairs...
Structures with usable C-alpha coordinates: 100

Generated controls for 112882 of 112882 observed pairs.
Saved to: /content/AMR_Structural_Mutation_Project/processed/matched_controls.csv

Match-type summary:
Match_type
exact    112882

Sample matched controls:
  Pair_id Structure_id  Control_Pos_1 Control_Insertion_Code_1  Control_Pos_2 Control_Insertion_Code_2  Control_Sequence_distance  Observed_Sequence_distance  Control_3D_distance Distance_unit Match_type                                                                 PDB_path
PAIR_0001 AF-Q9HV32-F1             29                                     115                                                  86                          86               18.959      angstrom      exact /content/AMR_Structural_Mutation_Project/raw/structures/AF-Q9HV32-F1.pdb
PAIR_0002 AF-H8ZYN4-F1             26                                      60                                                

In [17]:
import json
import numpy as np
import pandas as pd
from scipy import stats

# --------------------------------------------------
# 1. Define paths and load input files
# --------------------------------------------------
PROCESSED_DIR = BASE_DIR / "processed"

OBS_PATH = PROCESSED_DIR / "struct_distance_pairs.csv"
CONTROL_PATH = PROCESSED_DIR / "matched_controls.csv"
RESULTS_PATH = PROCESSED_DIR / "primary_statistical_results.json"

df_obs = pd.read_csv(OBS_PATH)
df_ctrl = pd.read_csv(CONTROL_PATH)

# Validate required columns
required_obs = {"Pair_id", "3D_distance"}
required_ctrl = {"Pair_id", "Control_3D_distance"}

missing_obs = required_obs - set(df_obs.columns)
missing_ctrl = required_ctrl - set(df_ctrl.columns)

if missing_obs:
    raise ValueError(
        f"Missing columns in struct_distance_pairs.csv: {sorted(missing_obs)}"
    )

if missing_ctrl:
    raise ValueError(
        f"Missing columns in matched_controls.csv: {sorted(missing_ctrl)}"
    )

# --------------------------------------------------
# 2. Check pair IDs and prepare a one-to-one comparison
# --------------------------------------------------
if df_obs["Pair_id"].duplicated().any():
    n_dup = int(df_obs["Pair_id"].duplicated().sum())
    raise ValueError(
        f"Observed data contain {n_dup} duplicate Pair_id entries. "
        "Resolve these before performing a one-to-one paired comparison."
    )

if df_ctrl["Pair_id"].duplicated().any():
    n_dup = int(df_ctrl["Pair_id"].duplicated().sum())
    raise ValueError(
        f"Control data contain {n_dup} duplicate Pair_id entries. "
        "Choose a prespecified control per observed pair or use a "
        "statistical method designed for multiple controls."
    )

df_merged = pd.merge(
    df_obs,
    df_ctrl,
    on="Pair_id",
    how="inner",
    validate="one_to_one",
    suffixes=("_observed", "_control")
)

n_obs = len(df_obs)
n_ctrl = len(df_ctrl)
n_matched = len(df_merged)

if n_matched == 0:
    raise ValueError(
        "No matching Pair_id values were found between observed and control data."
    )

# Report unmatched records rather than silently ignoring them
observed_only = len(set(df_obs["Pair_id"]) - set(df_ctrl["Pair_id"]))
control_only = len(set(df_ctrl["Pair_id"]) - set(df_obs["Pair_id"]))

print("Input and matching checks")
print(f"Observed records: {n_obs}")
print(f"Control records:  {n_ctrl}")
print(f"Matched pairs:    {n_matched}")
print(f"Observed records without controls: {observed_only}")
print(f"Control records without observations: {control_only}")

# Convert distances to numeric and exclude invalid pairs
df_merged["3D_distance"] = pd.to_numeric(
    df_merged["3D_distance"], errors="coerce"
)
df_merged["Control_3D_distance"] = pd.to_numeric(
    df_merged["Control_3D_distance"], errors="coerce"
)

valid_mask = (
    np.isfinite(df_merged["3D_distance"])
    & np.isfinite(df_merged["Control_3D_distance"])
    & (df_merged["3D_distance"] >= 0)
    & (df_merged["Control_3D_distance"] >= 0)
)

n_invalid = int((~valid_mask).sum())
df_merged = df_merged.loc[valid_mask].copy()

if len(df_merged) < 2:
    raise ValueError(
        "Fewer than two valid matched pairs remain; paired statistical "
        "testing is not appropriate."
    )

obs_dist = df_merged["3D_distance"].to_numpy(dtype=float)
ctrl_dist = df_merged["Control_3D_distance"].to_numpy(dtype=float)
diff = obs_dist - ctrl_dist

# --------------------------------------------------
# 3. Descriptive statistics
# --------------------------------------------------
obs_mean = float(np.mean(obs_dist))
obs_median = float(np.median(obs_dist))
ctrl_mean = float(np.mean(ctrl_dist))
ctrl_median = float(np.median(ctrl_dist))

diff_mean = float(np.mean(diff))
diff_median = float(np.median(diff))
diff_std = float(np.std(diff, ddof=1))
diff_se = float(stats.sem(diff))
n_pairs = len(diff)

# 95% confidence interval for the mean paired difference
t_critical = float(stats.t.ppf(0.975, df=n_pairs - 1))
diff_ci_low = diff_mean - t_critical * diff_se
diff_ci_high = diff_mean + t_critical * diff_se

# --------------------------------------------------
# 4. Paired statistical tests
# --------------------------------------------------
# Paired t-test: tests whether the mean paired difference is zero.
t_result = stats.ttest_rel(obs_dist, ctrl_dist, nan_policy="omit")

# Wilcoxon signed-rank test: nonparametric paired comparison.
# If all differences are zero, the test is undefined.
if np.allclose(diff, 0):
    wilcoxon_stat = None
    wilcoxon_pvalue = None
    wilcoxon_note = "Undefined because all paired differences are zero."
else:
    try:
        wilcoxon_result = stats.wilcoxon(
            obs_dist,
            ctrl_dist,
            alternative="two-sided",
            zero_method="wilcox",
            method="auto"
        )
        wilcoxon_stat = float(wilcoxon_result.statistic)
        wilcoxon_pvalue = float(wilcoxon_result.pvalue)
        wilcoxon_note = None
    except (ValueError, FloatingPointError) as exc:
        wilcoxon_stat = None
        wilcoxon_pvalue = None
        wilcoxon_note = str(exc)

# Cohen's dz: mean paired difference / SD of paired differences
cohens_dz = (
    diff_mean / diff_std
    if np.isfinite(diff_std) and diff_std > 0
    else None
)

# --------------------------------------------------
# 5. Save results
# --------------------------------------------------
results = {
    "analysis": "paired_observed_vs_matched_control_3D_distance",
    "distance_unit": "angstrom",
    "random_seed_used_for_control_generation": 42,
    "input_observed_records": int(n_obs),
    "input_control_records": int(n_ctrl),
    "matched_records_before_distance_validation": int(n_matched),
    "invalid_distance_pairs_excluded": int(n_invalid),
    "unmatched_observed_records": int(observed_only),
    "unmatched_control_records": int(control_only),
    "total_pairs_analyzed": int(n_pairs),
    "observed_3D_mean_A": obs_mean,
    "observed_3D_median_A": obs_median,
    "control_3D_mean_A": ctrl_mean,
    "control_3D_median_A": ctrl_median,
    "difference_mean_A": diff_mean,
    "difference_median_A": diff_median,
    "difference_SD_A": diff_std,
    "difference_SE_A": diff_se,
    "difference_95CI_low_A": float(diff_ci_low),
    "difference_95CI_high_A": float(diff_ci_high),
    "paired_ttest_statistic": float(t_result.statistic),
    "paired_ttest_pvalue": float(t_result.pvalue),
    "wilcoxon_statistic": wilcoxon_stat,
    "wilcoxon_pvalue": wilcoxon_pvalue,
    "wilcoxon_note": wilcoxon_note,
    "cohens_dz": cohens_dz,
    "interpretation_note": (
        "Negative mean difference indicates observed pairs have smaller "
        "C-alpha distances than their matched controls. These comparisons "
        "describe structural proximity and do not alone establish epistasis."
    )
}

with RESULTS_PATH.open("w", encoding="utf-8") as f:
    json.dump(results, f, indent=4, allow_nan=False)

# --------------------------------------------------
# 6. Print report
# --------------------------------------------------
print("\n=== STEP 12: PRIMARY STATISTICAL COMPARISON ===")
print(f"Valid paired observations: {n_pairs}")
print(
    f"Observed C-alpha distance (Å): "
    f"mean = {obs_mean:.3f}, median = {obs_median:.3f}"
)
print(
    f"Control C-alpha distance (Å):  "
    f"mean = {ctrl_mean:.3f}, median = {ctrl_median:.3f}"
)
print(f"Mean paired difference (observed - control): {diff_mean:.3f} Å")
print(f"Median paired difference: {diff_median:.3f} Å")
print(f"95% CI for mean difference: [{diff_ci_low:.3f}, {diff_ci_high:.3f}] Å")
print(
    f"Paired t-test: t = {t_result.statistic:.4f}, "
    f"p = {t_result.pvalue:.4e}"
)

if wilcoxon_pvalue is not None:
    print(
        f"Wilcoxon signed-rank test: statistic = {wilcoxon_stat:.4f}, "
        f"p = {wilcoxon_pvalue:.4e}"
    )
else:
    print(f"Wilcoxon signed-rank test unavailable: {wilcoxon_note}")

if cohens_dz is not None:
    print(f"Cohen's dz (paired effect size): {cohens_dz:.4f}")
else:
    print("Cohen's dz undefined because the paired-difference SD is zero.")

print(f"\nResults saved to: {RESULTS_PATH}")

Input and matching checks
Observed records: 112882
Control records:  112882
Matched pairs:    112882
Observed records without controls: 0
Control records without observations: 0

=== STEP 12: PRIMARY STATISTICAL COMPARISON ===
Valid paired observations: 112882
Observed C-alpha distance (Å): mean = 20.304, median = 18.454
Control C-alpha distance (Å):  mean = 22.883, median = 20.855
Mean paired difference (observed - control): -2.579 Å
Median paired difference: -1.667 Å
95% CI for mean difference: [-2.650, -2.509] Å
Paired t-test: t = -71.9765, p = 0.0000e+00
Wilcoxon signed-rank test: statistic = 2360500374.0000, p = 0.0000e+00
Cohen's dz (paired effect size): -0.2142

Results saved to: /content/AMR_Structural_Mutation_Project/processed/primary_statistical_results.json


In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --------------------------------------------------
# 1. Define paths and load data
# --------------------------------------------------
PROCESSED_DIR = BASE_DIR / "processed"
RESULTS_DIR = BASE_DIR / "results"
FIG_DIR = RESULTS_DIR / "figures"

FIG_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

OBS_PATH = PROCESSED_DIR / "struct_distance_pairs.csv"
CONTROL_PATH = PROCESSED_DIR / "matched_controls.csv"

df_obs = pd.read_csv(OBS_PATH)
df_ctrl = pd.read_csv(CONTROL_PATH)

required_obs = {"Pair_id", "3D_distance", "Sequence_distance"}
required_ctrl = {"Pair_id", "Control_3D_distance"}

missing_obs = required_obs - set(df_obs.columns)
missing_ctrl = required_ctrl - set(df_ctrl.columns)

if missing_obs:
    raise ValueError(
        f"Missing columns in struct_distance_pairs.csv: {sorted(missing_obs)}"
    )

if missing_ctrl:
    raise ValueError(
        f"Missing columns in matched_controls.csv: {sorted(missing_ctrl)}"
    )

if df_obs["Pair_id"].duplicated().any():
    raise ValueError(
        "Duplicate Pair_id values found in observed data. "
        "Resolve duplicates before plotting paired comparisons."
    )

if df_ctrl["Pair_id"].duplicated().any():
    raise ValueError(
        "Duplicate Pair_id values found in control data. "
        "Resolve duplicates before plotting paired comparisons."
    )

df_merged = pd.merge(
    df_obs,
    df_ctrl,
    on="Pair_id",
    how="inner",
    validate="one_to_one",
    suffixes=("_observed", "_control")
)

# Convert analysis columns to numeric
for col in ["3D_distance", "Sequence_distance", "Control_3D_distance"]:
    df_merged[col] = pd.to_numeric(df_merged[col], errors="coerce")

# Retain rows with valid, nonnegative distances and sequence separation
valid_mask = (
    np.isfinite(df_merged["3D_distance"])
    & np.isfinite(df_merged["Control_3D_distance"])
    & np.isfinite(df_merged["Sequence_distance"])
    & (df_merged["3D_distance"] >= 0)
    & (df_merged["Control_3D_distance"] >= 0)
    & (df_merged["Sequence_distance"] >= 0)
)

n_excluded = int((~valid_mask).sum())
df_merged = df_merged.loc[valid_mask].copy()

if df_merged.empty:
    raise ValueError(
        "No valid matched observations remain after filtering."
    )

print(f"Matched pairs available for plotting: {len(df_merged)}")
print(f"Rows excluded due to invalid values: {n_excluded}")

# --------------------------------------------------
# 2. Plot settings
# --------------------------------------------------
sns.set_theme(style="whitegrid", font_scale=1.1)

OBS_LABEL = "Observed pairs"
CTRL_LABEL = "Sequence-matched controls"

obs_median = df_merged["3D_distance"].median()
ctrl_median = df_merged["Control_3D_distance"].median()

# --------------------------------------------------
# 3. Plot 1: 3D distance distributions
# --------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 5))

obs_values = df_merged["3D_distance"].dropna()
ctrl_values = df_merged["Control_3D_distance"].dropna()

# KDE can fail or be misleading when values have zero variance.
if obs_values.nunique() > 1:
    sns.kdeplot(
        x=obs_values,
        label=OBS_LABEL,
        fill=True,
        alpha=0.30,
        linewidth=2,
        ax=ax
    )
else:
    ax.axvline(
        obs_values.iloc[0],
        label=f"{OBS_LABEL} (constant values)",
        linewidth=2
    )

if ctrl_values.nunique() > 1:
    sns.kdeplot(
        x=ctrl_values,
        label=CTRL_LABEL,
        fill=True,
        alpha=0.30,
        linewidth=2,
        ax=ax
    )
else:
    ax.axvline(
        ctrl_values.iloc[0],
        label=f"{CTRL_LABEL} (constant values)",
        linewidth=2,
        linestyle=":"
    )

ax.axvline(
    obs_median,
    linestyle="--",
    linewidth=1.5,
    label=f"Observed median ({obs_median:.2f} Å)"
)

ax.axvline(
    ctrl_median,
    linestyle="--",
    linewidth=1.5,
    label=f"Control median ({ctrl_median:.2f} Å)"
)

ax.set_title(
    "Cα Distance Distributions of Observed and Control Pairs",
    fontsize=12,
    fontweight="bold"
)
ax.set_xlabel("Cα–Cα distance (Å)")
ax.set_ylabel("Density")
ax.legend(frameon=True)
fig.tight_layout()

fig.savefig(
    FIG_DIR / "3d_distance_distribution.png",
    dpi=300,
    bbox_inches="tight"
)
plt.close(fig)

# --------------------------------------------------
# 4. Plot 2: Distance by sequence-separation bins
# --------------------------------------------------
# Bins represent sequence separation in residues:
# 1–10, 11–25, 26–50, 51–100, 101–200, >200.
#
# Include zero in the first bin for completeness, although
# distinct residue pairs should ordinarily have separation >= 1.
bins = [-np.inf, 10, 25, 50, 100, 200, np.inf]
labels = ["≤10", "11–25", "26–50", "51–100", "101–200", ">200"]

df_merged["Seq_Dist_Bin"] = pd.cut(
    df_merged["Sequence_distance"],
    bins=bins,
    labels=labels,
    right=True,
    ordered=True
)

# Calculate counts and mean distances by bin
bin_stats = (
    df_merged.groupby("Seq_Dist_Bin", observed=False)
    .agg(
        Pair_Count=("Pair_id", "count"),
        Observed_Mean_3D_A=("3D_distance", "mean"),
        Control_Mean_3D_A=("Control_3D_distance", "mean"),
        Observed_Median_3D_A=("3D_distance", "median"),
        Control_Median_3D_A=("Control_3D_distance", "median")
    )
    .reindex(labels)
)

bin_stats.index.name = "Seq_Bin"
bin_export = bin_stats.reset_index()

bin_export["Delta_Mean_A"] = (
    bin_export["Observed_Mean_3D_A"]
    - bin_export["Control_Mean_3D_A"]
)

# Save full stratification table
numeric_cols = [
    "Observed_Mean_3D_A",
    "Control_Mean_3D_A",
    "Observed_Median_3D_A",
    "Control_Median_3D_A",
    "Delta_Mean_A"
]
bin_export[numeric_cols] = bin_export[numeric_cols].round(3)

bin_export.to_csv(
    RESULTS_DIR / "sequence_bin_stratification.csv",
    index=False
)

# --------------------------------------------------
# 5. Plot grouped means by sequence separation
# --------------------------------------------------
x = np.arange(len(labels))
width = 0.36

means_obs = bin_stats["Observed_Mean_3D_A"].to_numpy(dtype=float)
means_ctrl = bin_stats["Control_Mean_3D_A"].to_numpy(dtype=float)

fig, ax = plt.subplots(figsize=(9, 5))

ax.bar(
    x - width / 2,
    means_obs,
    width,
    label=OBS_LABEL,
    alpha=0.85
)

ax.bar(
    x + width / 2,
    means_ctrl,
    width,
    label=CTRL_LABEL,
    alpha=0.85
)

ax.set_xlabel("Sequence separation (residues)")
ax.set_ylabel("Mean Cα–Cα distance (Å)")
ax.set_title(
    "Mean Structural Distance by Sequence Separation",
    fontsize=12,
    fontweight="bold"
)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend(frameon=True)

fig.tight_layout()
fig.savefig(
    FIG_DIR / "3d_distance_by_seq_bin.png",
    dpi=300,
    bbox_inches="tight"
)
plt.close(fig)

# --------------------------------------------------
# 6. Report outputs
# --------------------------------------------------
print("\nFigures saved to:", FIG_DIR)
print("Stratification table saved to:", RESULTS_DIR / "sequence_bin_stratification.csv")

print("\nSequence-separation stratification:")
print(bin_export.to_string(index=False))

Matched pairs available for plotting: 112882
Rows excluded due to invalid values: 0

Figures saved to: /content/AMR_Structural_Mutation_Project/results/figures
Stratification table saved to: /content/AMR_Structural_Mutation_Project/results/sequence_bin_stratification.csv

Sequence-separation stratification:
Seq_Bin  Pair_Count  Observed_Mean_3D_A  Control_Mean_3D_A  Observed_Median_3D_A  Control_Median_3D_A  Delta_Mean_A
    ≤10       15045              10.038             10.770                 9.200                9.989        -0.731
  11–25       15819              18.063             19.760                17.397               18.705        -1.697
  26–50       20261              17.940             20.080                16.756               19.216        -2.140
 51–100       29291              20.996             23.358                20.355               23.012        -2.363
101–200       22767              22.324             25.274                20.519               23.511        -2

In [19]:
import pandas as pd

# --------------------------------------------------
# 1. Define paths and load data
# --------------------------------------------------
PROCESSED_DIR = BASE_DIR / "processed"
RESULTS_DIR = BASE_DIR / "results"
FIG_DIR = RESULTS_DIR / "figures"

FIG_DIR.mkdir(parents=True, exist_ok=True)

PAIRS_PATH = PROCESSED_DIR / "struct_distance_pairs.csv"
STRUCTURE_MAP_PATH = PROCESSED_DIR / "structure_mapping.csv"

df_pairs = pd.read_csv(PAIRS_PATH)
df_struct = pd.read_csv(STRUCTURE_MAP_PATH)

required_pair_cols = {"Structure_id", "Pos_1", "Pos_2"}
required_map_cols = {"Structure_id", "PDB_path", "Chain"}

missing_pairs = required_pair_cols - set(df_pairs.columns)
missing_map = required_map_cols - set(df_struct.columns)

if missing_pairs:
    raise ValueError(
        f"Missing columns in struct_distance_pairs.csv: {sorted(missing_pairs)}"
    )

if missing_map:
    raise ValueError(
        f"Missing columns in structure_mapping.csv: {sorted(missing_map)}"
    )

# Remove rows without structure IDs or positions
df_pairs = df_pairs.dropna(
    subset=["Structure_id", "Pos_1", "Pos_2"]
).copy()

if df_pairs.empty:
    raise ValueError("No usable structure-distance records were found.")

# Ensure positions are numeric integers
for col in ["Pos_1", "Pos_2"]:
    df_pairs[col] = pd.to_numeric(df_pairs[col], errors="coerce")

df_pairs = df_pairs.dropna(subset=["Pos_1", "Pos_2"]).copy()
df_pairs = df_pairs[
    (df_pairs["Pos_1"] % 1 == 0)
    & (df_pairs["Pos_2"] % 1 == 0)
].copy()

df_pairs["Pos_1"] = df_pairs["Pos_1"].astype(int)
df_pairs["Pos_2"] = df_pairs["Pos_2"].astype(int)

if df_pairs.empty:
    raise ValueError("No valid integer residue positions remain.")

# --------------------------------------------------
# 2. Identify the top five structures
# --------------------------------------------------
top_structures = df_pairs["Structure_id"].value_counts().head(5)

print("=== TOP 5 STRUCTURES BY OBSERVED PAIR COUNT ===")
print(top_structures.to_string())

top_struct_id = top_structures.index[0]
df_top = df_pairs[df_pairs["Structure_id"] == top_struct_id].copy()

print(
    f"\nSelected structure for Figure 3: "
    f"{top_struct_id} ({len(df_top)} observed pairs)"
)

# --------------------------------------------------
# 3. Retrieve structure file and chain metadata
# --------------------------------------------------
map_rows = df_struct[
    df_struct["Structure_id"].astype(str) == str(top_struct_id)
]

if map_rows.empty:
    raise ValueError(
        f"No structure_mapping.csv entry found for {top_struct_id}."
    )

structure_info = map_rows.iloc[0]
pdb_path_value = structure_info["PDB_path"]
chain_value = structure_info["Chain"]

if pd.isna(pdb_path_value) or not str(pdb_path_value).strip():
    raise ValueError(f"No PDB_path recorded for structure {top_struct_id}.")

pdb_path = str(pdb_path_value)
chain_id = "A" if pd.isna(chain_value) else str(chain_value).strip()

# Resolve the path, including the project's structures directory
from pathlib import Path

pdb_file = Path(pdb_path)
if not pdb_file.exists():
    fallback_path = BASE_DIR / "raw" / "structures" / pdb_file.name
    if fallback_path.exists():
        pdb_file = fallback_path
    else:
        raise FileNotFoundError(
            f"Structure file not found: {pdb_path}\n"
            f"Also checked: {fallback_path}"
        )

# --------------------------------------------------
# 4. Collect unique residue positions
# --------------------------------------------------
all_positions = sorted(
    set(df_top["Pos_1"].tolist() + df_top["Pos_2"].tolist())
)

if not all_positions:
    raise ValueError("No residue positions are available for visualization.")

# PyMOL residue selection accepts + separated residue identifiers.
pos_selection_str = "+".join(map(str, all_positions))

# --------------------------------------------------
# 5. Generate PyMOL script
# --------------------------------------------------
# Quote paths for PyMOL and escape double quotes.
pdb_path_pml = str(pdb_file.resolve()).replace("\\", "/").replace('"', '\\"')
output_png = str(
    (FIG_DIR / "figure3_pymol_3d_render.png").resolve()
).replace("\\", "/").replace('"', '\\"')

pymol_script = f'''# Figure 3: 3D distribution of observed mutation-pair sites
# Structure ID: {top_struct_id}
# Source PDB: {pdb_path_pml}
# Chain: {chain_id}
#
# IMPORTANT:
# This script assumes Pos_1 and Pos_2 match PDB residue numbering.
# Confirm residue mapping before interpreting the visualization.

reinitialize
load "{pdb_path_pml}", target_structure

bg_color white
hide everything
show cartoon, target_structure
color gray80, target_structure

# Restrict the selection to the specified chain and residue positions
select mut_sites, target_structure and chain {chain_id} and resi {pos_selection_str}
show spheres, mut_sites
color orange, mut_sites

# Improve display
set sphere_scale, 0.8
set ray_opaque_background, on
set antialias, 2
set cartoon_fancy_helices, 1
set ray_trace_mode, 1
set ray_shadows, on

orient mut_sites
zoom mut_sites, 5

# Render and save the figure
png "{output_png}", width=2400, height=1800, dpi=300, ray=1
'''

pymol_path = FIG_DIR / "render_figure3.pml"
pymol_path.write_text(pymol_script, encoding="utf-8")

# Export selected structure's positions for reproducibility
positions_path = FIG_DIR / "figure3_highlighted_positions.csv"

pd.DataFrame({
    "Structure_id": top_struct_id,
    "Chain": chain_id,
    "Residue_position": all_positions
}).to_csv(positions_path, index=False)

print(f"\nPyMOL script saved to: {pymol_path}")
print(f"Highlighted-position table saved to: {positions_path}")
print(f"Structure file: {pdb_file}")
print(f"Chain: {chain_id}")
print(f"Unique residue positions: {len(all_positions)}")
print("\nResidue positions:")
print(pos_selection_str)

print(
    "\nTo generate the figure, open render_figure3.pml in PyMOL. "
    "The PNG will be written to results/figures/."
)

=== TOP 5 STRUCTURES BY OBSERVED PAIR COUNT ===
Structure_id
AF-I6XD65-F1    80467
AF-P9WIE5-F1    15486
AF-P9WGY9-F1    10974
AF-P9WNL7-F1     1162
AF-P9WNF9-F1      737

Selected structure for Figure 3: AF-I6XD65-F1 (80467 observed pairs)

PyMOL script saved to: /content/AMR_Structural_Mutation_Project/results/figures/render_figure3.pml
Highlighted-position table saved to: /content/AMR_Structural_Mutation_Project/results/figures/figure3_highlighted_positions.csv
Structure file: /content/AMR_Structural_Mutation_Project/raw/structures/AF-I6XD65-F1.pdb
Chain: A
Unique residue positions: 156

Residue positions:
1+3+4+5+6+7+8+9+10+11+12+13+14+16+17+18+19+21+22+23+24+26+27+28+29+30+31+32+34+35+36+38+41+42+43+44+45+46+47+48+49+51+52+53+54+55+57+58+59+61+62+63+64+65+66+67+68+69+70+71+72+75+76+77+78+79+80+81+82+85+86+88+89+90+92+93+94+95+96+97+100+101+102+103+104+105+106+108+109+110+112+114+115+116+117+118+119+120+121+122+123+124+125+127+128+129+130+131+132+133+134+135+136+137+138+139+140+141